# Python 클래스_전체 정리

**한 줄 요약** · 클래스 수업 노트북 4개를 **중복 없이 4개 대단원 · 12개 섹션**으로 재구성하고, 마지막 **종합 실습**에서 PyTorch `Dataset` · `nn.Module`로 연결

- 정답 코드 · 실행 결과 · 해설 포함
- 딥러닝 코드의 `class` 부분을 읽기 위한 최소 문법

## 🎯 학습 목표

이 노트북을 끝내면 **자료를 보지 않고** 다음을 할 수 있어야 함

1. `class` · `__init__` · `self`로 설계도를 만들고, 객체마다 다른 값과 공유 값을 구분해 저장한다
2. `len()` · `[i]` · `print()` · `객체(x)`가 부르는 특수 메서드를 말하고 직접 정의한다
3. `class 자식(부모)`로 물려받아 메서드를 덮어쓰고, `super()`로 부모 기능에 덧붙인다
4. `Dataset`을 상속해 `__len__` · `__getitem__`을 채우고 `DataLoader`로 배치를 만든다
5. `nn.Module`을 상속해 `super().__init__()` · `forward`를 쓰고, 모델을 `model(x)`로 부르는 이유를 설명한다

## 어디에 쓰나

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">실제 상황</th>
<th style="text-align: left;">이 권에서 배우는 것</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">이미지 · 라벨 목록으로 학습 데이터를 공급할 때</td>
<td style="text-align: left;"><code>Dataset</code>의 <code>__len__</code> · <code>__getitem__</code> (3-3 · 5장)</td>
</tr>
<tr>
<td style="text-align: left;">모델 클래스를 직접 만들 때</td>
<td style="text-align: left;"><code>nn.Module</code> 상속 · <code>super().__init__()</code> · <code>forward</code> (4-3 · 5장)</td>
</tr>
<tr>
<td style="text-align: left;"><code>cannot assign module before Module.__init__() call</code> 에러를 만났을 때</td>
<td style="text-align: left;"><code>super().__init__()</code> 순서 (4-3 · 5장)</td>
</tr>
<tr>
<td style="text-align: left;"><code>print(model)</code>로 층 구조를 볼 때</td>
<td style="text-align: left;"><code>__repr__</code> (3-2)</td>
</tr>
<tr>
<td style="text-align: left;">학습 루프에서 <code>model(x)</code> · <code>loss_fn(out, y)</code>를 부를 때</td>
<td style="text-align: left;"><code>__call__</code> → <code>forward</code> (3-4)</td>
</tr>
<tr>
<td style="text-align: left;">데이터셋 객체끼리 값이 섞이는 버그를 만났을 때</td>
<td style="text-align: left;">인스턴스 변수 vs 클래스 변수 (2-2)</td>
</tr>
</tbody>
</table>

## 학습 지도

<img src="images/cls2_map.png" alt="Python 클래스 학습 지도" width="763">

**예상 학습 시간** · 개념 2시간 + 실습 1시간 (5장 PyTorch도 CPU에서 몇 초 안에 실행됨)

## 목차

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: center;">장</th>
<th style="text-align: left;">제목</th>
<th style="text-align: left;">이 장의 질문</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: center;">0</td>
<td style="text-align: left;">준비</td>
<td style="text-align: left;">실행 순서 · 5장에 필요한 PyTorch 설치 확인</td>
</tr>
<tr>
<td style="text-align: center;">1</td>
<td style="text-align: left;">클래스가 필요한 이유</td>
<td style="text-align: left;">데이터를 리스트 · 딕셔너리로 흩어 두면 무엇이 문제이고, 클래스는 어떻게 묶는가?</td>
</tr>
<tr>
<td style="text-align: center;">2</td>
<td style="text-align: left;">self · 변수</td>
<td style="text-align: left;"><code>self</code>는 무엇이며, 객체마다 따로인 값과 공유하는 값은 어떻게 나누는가?</td>
</tr>
<tr>
<td style="text-align: center;">3</td>
<td style="text-align: left;">특수 메서드</td>
<td style="text-align: left;"><code>print()</code> · <code>len()</code> · <code>[i]</code> · <code>객체(x)</code>를 내가 만든 객체에 쓰려면?</td>
</tr>
<tr>
<td style="text-align: center;">4</td>
<td style="text-align: left;">상속 · super</td>
<td style="text-align: left;">이미 있는 클래스를 물려받아 일부만 바꾸고 덧붙이려면?</td>
</tr>
<tr>
<td style="text-align: center;">5</td>
<td style="text-align: left;">종합 실습</td>
<td style="text-align: left;">PyTorch <code>Dataset</code> · <code>nn.Module</code>은 1~4장의 어떤 문법으로 만들어지는가?</td>
</tr>
<tr>
<td style="text-align: center;">—</td>
<td style="text-align: left;">마무리 · 부록</td>
<td style="text-align: left;">오늘 배운 것 · 기억 확인 · 오답 노트 · 면접 · 요약표 · 복습일 · 원본 대응표 · 전체 코드</td>
</tr>
</tbody>
</table>

---
## 0. 준비

1 ~ 4장은 파이썬만, 5장은 PyTorch가 필요함. **셀은 위에서부터 차례로 실행**(앞 섹션의 `car1` · `Animal` 등을 이어서 씀).

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">항목</th>
<th style="text-align: left;">설정</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">실행 순서</td>
<td style="text-align: left;">위에서부터 차례로 (3-1 · 3-3 · 3-4는 1-2의 <code>car1</code>, 4-2 · 4-3은 4-1의 <code>Animal</code>을 씀)</td>
</tr>
<tr>
<td style="text-align: left;">라이브러리</td>
<td style="text-align: left;">1 ~ 4장 없음 · 5장 <code>torch</code></td>
</tr>
<tr>
<td style="text-align: left;">seed</td>
<td style="text-align: left;">5장 가중치 초기값만 <code>torch.manual_seed(0)</code> (출력은 모양 · 개수라 seed와 무관)</td>
</tr>
<tr>
<td style="text-align: left;">실행 장치</td>
<td style="text-align: left;">CPU로 충분 (이미지 12장 · 8×8 가짜 데이터)</td>
</tr>
<tr>
<td style="text-align: left;">데이터</td>
<td style="text-align: left;">없음 · 5장에서 가짜 이미지를 만들어 씀</td>
</tr>
</tbody>
</table>

In [1]:
# 버전 확인: 1 ~ 4장은 파이썬만, 5장은 PyTorch 필요
import sys             # 파이썬 버전 확인용
import importlib.util  # 설치 여부만 확인 (불러오지는 않음)

print("Python:", sys.version.split()[0])                                 # 3.9 이상이면 됨
print("PyTorch 설치됨:", importlib.util.find_spec("torch") is not None)  # False면 5장 실행 불가

Python: 3.14.6
PyTorch 설치됨: True


---
## 1. 클래스가 필요한 이유

> **이 장의 흐름**
>
> - 01_Python문법에서 값을 리스트 · 딕셔너리로 묶었다
> - 자동차처럼 같은 구조의 물건이 많아지고 기능까지 붙으면, 데이터와 기능을 한 덩어리로 묶는 틀(클래스)이 필요

**파이프라인 위치** · ② 불러오기 · ④ 모델 — 데이터셋 · 모델은 모두 클래스로 만든 객체

### 1-1. `B` 흩어진 데이터의 문제 · `list` · `dict` <sub>원본 01 · 1~5절</sub>

> 🎯 **자동차 정보를 변수 · 리스트 · 딕셔너리로 관리하면 어떤 문제가 생기는가?**

💡 **쉽게 말하면**

- 리스트 두 개로 나누면 **이름표와 물건을 다른 상자에 보관**한 것.
- 한쪽 상자만 정리하면 이름표가 엉뚱한 물건에 붙음.

**실행 흐름**

1. 리스트 두 개
2. 한쪽만 삭제
3. zip 짝 어긋남
4. 딕셔너리로 묶기
5. 키 오타 KeyError

**먼저 예측** · 회사 목록에서만 BMW를 지운 뒤 `zip`으로 짝지으면 `Audi` 옆에는 누구의 정보가 붙나? — 답은 코드 실행 결과 아래에

In [2]:
# 방법 1: 회사 · 상세 정보를 리스트 두 개에 나눠 저장
companies = ["Ferrari", "BMW", "Audi"]                            # 회사 이름만 모은 리스트
details = [{"color": "White", "horsepower": 400, "price": 8000},  # 상세 정보는 다른 리스트에
           {"color": "Black", "horsepower": 270, "price": 5000},
           {"color": "Silver", "horsepower": 300, "price": 6000}]


# 회사 목록에서만 BMW를 지우고 짝짓기
del companies[1]                                 # details는 그대로 3개
for company, detail in zip(companies, details):  # 짧은 쪽(2개)에 맞춰 같은 위치끼리
    print(company, detail)                       # Audi 옆에 누구 정보가 붙나


# 방법 2: 한 대의 정보를 딕셔너리 하나로 묶기
# 한 대 = 딕셔너리 하나
cars = [{"company": "Ferrari", "color": "White", "horsepower": 400, "price": 8000},
        {"company": "BMW", "color": "Black", "horsepower": 270, "price": 5000},
        {"company": "Audi", "color": "Silver", "horsepower": 300, "price": 6000}]
del cars[1]                                          # 한 대 = 한 덩어리 → 정보가 함께 지워짐
print("남은 차:", [car["company"] for car in cars])  # 회사 이름만 꺼내 확인


# 딕셔너리의 남은 문제: 키 오타(horse_power)는 실행해야 드러남
# 키 이름 오타
wrong_car = {"company": "Ferrari", "color": "White", "horse_power": 400, "price": 8000}
try:
    print(wrong_car["horsepower"])  # 없는 키로 꺼내기
except KeyError as e:               # 없는 키 → KeyError
    print("KeyError: 없는 키", e)   # 어느 키인지 출력

Ferrari {'color': 'White', 'horsepower': 400, 'price': 8000}
Audi {'color': 'Black', 'horsepower': 270, 'price': 5000}
남은 차: ['Ferrari', 'Audi']
KeyError: 없는 키 'horsepower'


**결과** · 예측 정답은 BMW의 정보(`Black` · `270`) — 에러 없이 조용히 어긋남

- `Audi {'color': 'Black', 'horsepower': 270, 'price': 5000}` — Audi 옆에 BMW 정보 → 에러 없는 버그
- `남은 차: ['Ferrari', 'Audi']` — 한 대 = 딕셔너리 하나 → 정보가 함께 지워짐
- `KeyError: 없는 키 'horsepower'` — 키 이름 오타는 실행해야 발견됨

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">변수 여러 개 <code>car_company_1</code></td>
<td style="text-align: left;">차 1대 = 변수 여러 개, 100대면 변수 수백 개</td>
<td style="text-align: left;">값이 몇 개뿐인 짧은 실험</td>
</tr>
<tr>
<td style="text-align: left;">리스트 여러 개 (같은 위치끼리 짝)</td>
<td style="text-align: left;">삭제 · 정렬을 <strong>모든 리스트에 똑같이</strong> 해야 함</td>
<td style="text-align: left;">길이가 고정된 읽기 전용 데이터</td>
</tr>
<tr>
<td style="text-align: left;">딕셔너리 리스트</td>
<td style="text-align: left;">한 대 = 한 덩어리, 키 오타는 <strong>실행해야</strong> 발견</td>
<td style="text-align: left;">JSON처럼 구조만 담아 저장 · 전달할 때</td>
</tr>
<tr>
<td style="text-align: left;">클래스</td>
<td style="text-align: left;">데이터 + <strong>기능(메서드)</strong> 함께, 속성 이름이 코드에 고정</td>
<td style="text-align: left;">같은 구조의 물건이 많고 동작도 필요할 때</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 에러 없이 결과만 틀리는 버그가 가장 위험 · 이미지 목록
> - 라벨 목록을 따로 섞거나 지울 때 같은 일이 생김

> ✅ **한 줄 정리** · 관련된 값은 한 덩어리로 묶어야 하며, 데이터와 기능을 함께 묶는 틀이 클래스다.

**딥러닝 코드에서는**

- `image, label = dataset[i]`
- 이미지와 라벨을 따로 두지 않고 **한 쌍**으로 돌려주는 이유

### 1-2. `A` 설계도와 객체 · `class` · `__init__` <sub>원본 01 · 6~9절</sub>

> 🎯 **클래스로 설계도를 만들고, 그 설계도로 객체를 어떻게 찍어 내는가?**

💡 **쉽게 말하면**

- 클래스는 **붕어빵 틀**, 객체는 **붕어빵**.
- `__init__`은 틀로 붕어빵을 찍는 순간 자동으로 실행되는 속 채우기.

<img src="images/cls2_class_object.png" alt="클래스 하나로 객체 여러 개" width="763">

**실행 흐름**

1. class Car 정의
2. Car(...) 3번
3. `__init__` 자동 실행
4. 속성 · `__dict__` 확인
5. 리스트 반복으로 메서드 호출

**먼저 예측** · `Car("BMW", "Black", 270, 5000)`에서 직접 부르지 않았는데 실행되는 메서드는? — 답은 코드 실행 결과 아래에

In [3]:
# 설계도(클래스) 정의: 데이터(속성) + 기능(메서드)
class Car:                                            # 설계도 이름 (대문자로 시작)
    """자동차 한 대의 데이터와 기능을 묶은 클래스"""  # 클래스 설명 (docstring)

    def __init__(self, company, color, horsepower, price):  # Car(...) 할 때 자동 실행
        self.company = company                              # self.이름 = 값 → 이 객체에 저장
        self.color = color                                  # 색
        self.horsepower = horsepower                        # 마력
        self.price = price                                  # 가격

    def is_high_performance(self):     # 메서드 = 클래스 안의 함수, 첫 인자는 self
        return self.horsepower >= 300  # 300마력 이상이면 True

    def apply_discount(self, rate):                     # 1-3에서 사용
        if not 0 <= rate <= 1:                          # 0~1 밖이면 → 에러 발생
            raise ValueError("할인율은 0 이상 1 이하")  # 잘못된 값은 여기서 멈춤
        self.price = round(self.price * (1 - rate))     # 이 객체의 price만 변경


# 설계도로 객체 3개 만들기
car1 = Car("Ferrari", "White", 400, 8000)  # Car(...) → 새 객체 생성 → __init__(새 객체, ...)
car2 = Car("BMW", "Black", 270, 5000)      # 같은 설계도로 두 번째 객체
car3 = Car("Audi", "Silver", 300, 6000)    # 세 번째 객체


# 속성 꺼내기 · 객체 안에 저장된 값 전체 보기
print(car1.company, car1.horsepower)  # 객체.속성
print(car2.__dict__)                  # {속성 이름: 값}


# 객체를 리스트로 관리하며 메서드 호출
car_list = [car1, car2, car3]  # 객체도 리스트에 담을 수 있음
for car in car_list:           # 객체마다 같은 메서드 호출
    print(f"{car.company}: {'고성능' if car.is_high_performance() else '일반'}")  # 객체.메서드()


# print(객체)는 아직 알아보기 어려움 → 3-2에서 해결
print("print(car1) →", str(car1).split(" at ")[0], "at 0x...>")  # 주소는 실행마다 달라 생략

Ferrari 400
{'company': 'BMW', 'color': 'Black', 'horsepower': 270, 'price': 5000}
Ferrari: 고성능
BMW: 일반
Audi: 고성능
print(car1) → <__main__.Car object at 0x...>


**결과** · 예측 정답은 `__init__` — 객체를 만들 때 자동 실행 (만들어진 객체가 `self`로 들어감)

- `Ferrari 400` — `객체.속성`으로 값 꺼내기
- `{'company': 'BMW', 'color': 'Black', 'horsepower': 270, 'price': 5000}` — `__dict__` = 그 객체에 저장된 속성 전부
- `Ferrari: 고성능` · `BMW: 일반` · `Audi: 고성능` — `horsepower >= 300`만 고성능 (원본 코드의 `>= 200`은 원본 설명 · 출력과 맞지 않아 `>= 300`으로 수정)
- `print(car1) → <__main__.Car object at 0x...>` — 출력 방법을 정하지 않으면 클래스 이름 + 메모리 주소 → 3-2에서 해결

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">클래스 <code>Car</code></td>
<td style="text-align: left;"><strong>설계도</strong> 1개</td>
<td style="text-align: left;">구조와 기능을 정의할 때</td>
</tr>
<tr>
<td style="text-align: left;">객체 <code>car1 = Car(...)</code></td>
<td style="text-align: left;">설계도로 만든 <strong>실물</strong>, 여러 개 가능</td>
<td style="text-align: left;">실제 값을 담아 쓸 때</td>
</tr>
<tr>
<td style="text-align: left;">속성 <code>car1.price</code></td>
<td style="text-align: left;"><strong>값</strong> · 괄호 없음</td>
<td style="text-align: left;">상태 읽기 · 저장</td>
</tr>
<tr>
<td style="text-align: left;">메서드 <code>car1.is_high_performance()</code></td>
<td style="text-align: left;"><strong>기능</strong> · 괄호로 호출</td>
<td style="text-align: left;">상태로 계산 · 상태 변경</td>
</tr>
<tr>
<td style="text-align: left;"><code>__init__</code></td>
<td style="text-align: left;">객체를 만들 때 <strong>자동</strong> 실행</td>
<td style="text-align: left;">속성 초기값 저장 (직접 호출 X)</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - `__init__`의 첫 인자 `self`는 호출할 때 넣지 않음
> - `Car("BMW", ...)`처럼 나머지 인자만 전달

> ✅ **한 줄 정리** · `class`로 속성과 메서드를 묶은 설계도를 만들고, `클래스(...)`로 부르면 `__init__`이 실행되어 객체가 만들어진다.

**딥러닝 코드에서는**

- `model = PetClassifier()` · `dataset = PetDataset(items)` · 모델
- 데이터셋도 클래스로 만든 객체 (5장)

### 1-3. `A` 상태를 바꾸는 메서드 · `메서드` · `raise` <sub>원본 01 · 10절</sub>

> 🎯 **메서드로 객체 하나의 값만 바꾸고, 잘못된 값은 어떻게 막는가?**

💡 **쉽게 말하면**

- 메서드는 **물건에 달린 버튼**.
- 버튼(`apply_discount`)을 누르면 그 물건의 값만 바뀌고, 이상한 입력이면 `raise`로 작동을 거부.

**실행 흐름**

1. 할인 전 확인
2. car1만 10% 할인
3. 1.5 할인
4. ValueError
5. 직접 수정은 막지 못함

**먼저 예측** · `car1.apply_discount(0.1)` 뒤 `car2.price`는? — 답은 코드 실행 결과 아래에

In [4]:
# 할인 전 가격
print("할인 전:", car1.price, car2.price)  # 두 객체의 처음 가격


# car1만 10% 할인
car1.apply_discount(0.1)                   # 8000 × 0.9 → round → 7200
print("할인 후:", car1.price, car2.price)  # car2는 그대로


# 범위 밖 할인율 → 직접 만든 에러
try:
    car1.apply_discount(1.5)     # 0~1 밖의 할인율
except ValueError as e:          # raise가 만든 에러 잡기
    print("ValueError:", e)      # 에러 문장 출력
print("가격 유지:", car1.price)  # raise에서 멈춰 price 변경 줄까지 가지 않음


# 메서드를 거치지 않고 밖에서 직접 바꾸면 검증을 건너뜀
car3.price = -100                # 막는 장치 없음
print("직접 수정:", car3.price)  # -100이 그대로 들어감
car3.price = 6000                # 다음 섹션을 위해 원래대로

할인 전: 8000 5000
할인 후: 7200 5000
ValueError: 할인율은 0 이상 1 이하
가격 유지: 7200
직접 수정: -100


**결과** · 예측 정답은 `5000` 그대로 — 바뀌는 것은 메서드를 호출한 `car1`뿐

- `할인 전: 8000 5000` · `할인 후: 7200 5000` — 8000 × 0.9 = 7200, car2는 그대로
- `ValueError: 할인율은 0 이상 1 이하` — `raise`로 직접 만든 에러
- `가격 유지: 7200` — 에러가 난 줄에서 멈춰 가격은 안 바뀜
- `직접 수정: -100` — 속성에 직접 대입하면 검증을 건너뜀

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">메서드 <code>car1.apply_discount(0.1)</code></td>
<td style="text-align: left;">검증 · 계산 규칙을 <strong>한 곳</strong>에서 적용</td>
<td style="text-align: left;">값을 바꾸는 기본 방법</td>
</tr>
<tr>
<td style="text-align: left;">직접 대입 <code>car1.price = 7200</code></td>
<td style="text-align: left;">검증 없이 <strong>바로</strong> 바뀜</td>
<td style="text-align: left;">규칙이 필요 없는 단순 값</td>
</tr>
<tr>
<td style="text-align: left;"><code>raise ValueError(...)</code></td>
<td style="text-align: left;">잘못된 입력에서 <strong>즉시 멈춤</strong></td>
<td style="text-align: left;">계속 진행하면 결과가 틀어질 때</td>
</tr>
<tr>
<td style="text-align: left;"><code>print()</code>로 경고만</td>
<td style="text-align: left;">실행은 계속 → 잘못된 값이 그대로 남음</td>
<td style="text-align: left;">디버깅 확인용 (검증 수단 X)</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 속성을 밖에서 직접 바꾸면 메서드의 검증이 소용없음
> - 값 변경은 메서드로

> ✅ **한 줄 정리** · 메서드는 `self`로 자기 객체의 값만 바꾸며, 잘못된 입력은 `raise`로 막는다.

**딥러닝 코드에서는**

- `optimizer.step()` · `model.train()`
- 객체의 상태(가중치 · 모드)를 메서드로 바꾸는 같은 구조

---
## 2. self · 인스턴스 변수 · 클래스 변수

> **이 장의 흐름**
>
> - 1장에서 같은 클래스로 객체를 여러 개 만들었다
> - 메서드가 '어느 객체'의 값을 쓰는지(self), 값이 객체마다 따로인지 공유인지 구분해야 함

**파이프라인 위치** · ④ 모델 — `self.fc` · `self.items`처럼 객체가 기억하는 값

### 2-1. `A` self의 정체 · `self` <sub>원본 02 · 1~3절</sub>

> 🎯 **메서드 정의에는 있는 `self`를 호출할 때는 왜 넣지 않는가?**

💡 **쉽게 말하면**

- `self`는 **'이 버튼을 누른 바로 그 물건'**.
- `car1.메서드()`라고 부르면 파이썬이 `car1`을 `self` 자리에 자동으로 넣어 줌.

<img src="images/cls2_self.png" alt="객체.메서드 호출은 클래스.메서드에 객체를 넣는 것과 같음" width="763">

**실행 흐름**

1. 두 호출 방식 비교
2. self가 호출한 객체인지 확인
3. self 빠뜨린 정의 TypeError
4. self. 빠뜨린 저장

**먼저 예측** · `def hello():`처럼 `self` 없이 정의한 메서드를 `객체.hello()`로 부르면? — 답은 코드 실행 결과 아래에

In [5]:
# 같은 메서드를 두 방식으로 호출
print(car1.is_high_performance(), Car.is_high_performance(car1))  # 객체.m() = 클래스.m(객체)


# self 자리에 무엇이 들어오는지 확인
class Probe:         # self가 무엇인지 보는 실험용 클래스
    def who(self):   # 첫 인자 self = 호출한 객체
        return self  # 받은 self를 그대로 돌려줌


p1, p2 = Probe(), Probe()                                                      # 객체 두 개
print("p1.who() is p1:", p1.who() is p1, "/ p2.who() is p2:", p2.who() is p2)  # 호출한 객체 = self


# self를 빠뜨리고 정의하면
class NoSelf:        # self를 빠뜨린 클래스
    def hello():     # self 자리 없음
        return "hi"  # 여기까지 오지 못함


try:
    NoSelf().hello()        # 파이썬은 호출한 객체를 자동으로 1개 넘김
except TypeError as e:      # 인자 개수가 안 맞음
    print("TypeError:", e)  # 에러 문장 그대로


# self. 를 빠뜨리고 저장하면
class Forget:                  # self. 를 빠뜨린 클래스
    def __init__(self, name):  # 이름을 받지만
        name = name            # 지역 변수에 저장 → 함수가 끝나면 사라짐


print("Forget 객체에 저장된 값:", Forget("Kim").__dict__)  # 객체에 남은 값 확인

True True
p1.who() is p1: True / p2.who() is p2: True
TypeError: NoSelf.hello() takes 0 positional arguments but 1 was given
Forget 객체에 저장된 값: {}


**결과** · 예측 정답은 `TypeError` — 파이썬이 객체 1개를 자동으로 넘기는데 받을 자리가 없음

- `True True` — `car1.is_high_performance()` = `Car.is_high_performance(car1)`
- `p1.who() is p1: True / p2.who() is p2: True` — self = 메서드를 호출한 그 객체 (`is`는 01_Python문법 1-2)
- `TypeError: NoSelf.hello() takes 0 positional arguments but 1 was given` — 0개를 받는데 1개(객체)가 넘어옴
- `Forget 객체에 저장된 값: {}` — `self.` 없이 저장 → 지역 변수라 사라짐

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>car1.method()</code></td>
<td style="text-align: left;">객체가 <strong>자동으로</strong> self에 들어감</td>
<td style="text-align: left;"><strong>기본 선택</strong></td>
</tr>
<tr>
<td style="text-align: left;"><code>Car.method(car1)</code></td>
<td style="text-align: left;">self 자리에 객체를 <strong>직접</strong> 넣음</td>
<td style="text-align: left;">동작 원리 확인용</td>
</tr>
<tr>
<td style="text-align: left;"><code>self.name = name</code></td>
<td style="text-align: left;"><strong>객체에</strong> 저장 → 다른 메서드에서도 사용</td>
<td style="text-align: left;">객체가 기억해야 하는 값</td>
</tr>
<tr>
<td style="text-align: left;"><code>name = name</code></td>
<td style="text-align: left;"><strong>지역 변수</strong> → 메서드가 끝나면 사라짐</td>
<td style="text-align: left;">메서드 안에서만 쓰는 계산용 값</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - `self.name = name`의 `self.`를 빠뜨리면 에러 없이 값이 사라짐
> - 나중에 쓸 때 `AttributeError`

> ✅ **한 줄 정리** · `self`는 메서드를 호출한 객체이며, `객체.메서드()`는 `클래스.메서드(객체)`로 바뀌어 실행된다.

**딥러닝 코드에서는**

- `def forward(self, x): return self.fc(x)`
- `self.fc`는 이 모델 객체가 가진 층

### 2-2. `B` 인스턴스 변수와 클래스 변수 · `self.x` · `클래스.x` <sub>원본 02 · 4~9절</sub>

> 🎯 **객체마다 따로 갖는 값과 모든 객체가 함께 쓰는 값은 어떻게 나누는가?**

💡 **쉽게 말하면**

- 인스턴스 변수는 **각자의 사물함**, 클래스 변수는 **반 전체 게시판**.
- 게시판은 반 이름(`Car.`)으로 고쳐야 하고, 내 사물함에 같은 제목 쪽지를 넣으면 나만 그 쪽지를 보게 됨.

<img src="images/cls2_variables.png" alt="인스턴스 변수는 객체마다, 클래스 변수는 하나를 공유" width="763">

**실행 흐름**

1. 객체 2개 생성
2. 가격 독립 확인
3. 생성 수 공유 확인
4. 객체 이름으로 대입 · 삭제
5. 리스트 클래스 변수

**먼저 예측** · `a.total_created = 100` 뒤 `Car.total_created`는? — 답은 코드 실행 결과 아래에

In [6]:
# 생성 수를 세는 Car (1-2의 Car를 줄여 새로 정의)
class Car:             # 생성 수를 세는 단순한 Car
    total_created = 0  # 클래스 변수: 클래스에 1개 → 모든 객체가 공유

    def __init__(self, company, price):  # 객체마다 회사 · 가격
        self.company = company           # 인스턴스 변수: 객체마다 따로
        self.price = price               # 인스턴스 변수
        Car.total_created += 1           # 클래스 이름으로 공유 값 변경


a = Car("Ferrari", 8000)  # 생성 1 → total 1
b = Car("BMW", 5000)      # 생성 2 → total 2


# 인스턴스 변수는 객체마다 독립
a.price = 7500                      # a의 가격만 바꿈
print("price →", a.price, b.price)  # b는 그대로인지


# 클래스 변수는 모두 같은 값을 봄
print("total →", Car.total_created, a.total_created, b.total_created)  # 세 곳 모두 같은 값


# 객체 이름으로 대입하면 → 같은 이름의 인스턴스 변수가 새로 생김
a.total_created = 100                                                    # 객체 이름으로 대입
print("대입 후 →", Car.total_created, a.total_created, b.total_created)  # a만 달라짐
print("a에 저장된 값:", a.__dict__)                                      # a에 새로 생긴 변수 확인


# 인스턴스 변수를 지우면 다시 클래스 변수가 보임
del a.total_created                  # a의 인스턴스 변수 삭제
print("삭제 후 →", a.total_created)  # 객체에서 못 찾음 → 클래스에서 찾음


# 클래스 변수에 리스트를 두면
class Bag:      # 리스트를 클래스 변수로 둔 예
    items = []  # 클래스 변수 (모든 Bag이 공유)


x, y = Bag(), Bag()            # 객체 두 개
x.items.append("apple")        # 대입이 아니라 수정 → 클래스의 리스트가 바뀜
print("y.items도 →", y.items)  # x만 고쳤는데 y 확인

price → 7500 5000
total → 2 2 2
대입 후 → 2 100 2
a에 저장된 값: {'company': 'Ferrari', 'price': 7500, 'total_created': 100}
삭제 후 → 2
y.items도 → ['apple']


**결과** · 예측 정답은 `2` 그대로 — `a`에 같은 이름의 인스턴스 변수가 새로 생겼을 뿐

- `price → 7500 5000` · `total → 2 2 2` — 인스턴스 변수는 객체마다 독립 · 클래스 변수 1개를 모두가 봄 (원본의 큰 숫자는 생성 셀을 반복 실행해 누적된 값)
- `대입 후 → 2 100 2` · `'total_created': 100}` — a만 100 · a의 `__dict__`에 total_created가 새로 생김
- `삭제 후 → 2` — 객체에서 못 찾으면 클래스에서 찾음
- `y.items도 → ['apple']` — 리스트는 대입이 아니라 수정 → 공유 리스트가 바뀜

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">인스턴스 변수 <code>self.price</code></td>
<td style="text-align: left;"><strong>객체마다</strong> 따로 저장 (<code>객체.__dict__</code>)</td>
<td style="text-align: left;"><strong>기본 선택</strong> — 이름 · 가격 · 데이터 목록</td>
</tr>
<tr>
<td style="text-align: left;">클래스 변수 <code>Car.total_created</code></td>
<td style="text-align: left;">클래스에 <strong>1개</strong>, 모든 객체가 공유</td>
<td style="text-align: left;">생성 수 · 공통 상수</td>
</tr>
<tr>
<td style="text-align: left;"><code>객체.변수 = 값</code> (대입)</td>
<td style="text-align: left;">같은 이름의 <strong>인스턴스 변수 생성</strong>, 클래스 변수는 그대로</td>
<td style="text-align: left;">그 객체만 다른 값을 줄 때</td>
</tr>
<tr>
<td style="text-align: left;"><code>객체.리스트.append()</code> (수정)</td>
<td style="text-align: left;"><strong>공유 리스트 자체</strong>가 바뀜</td>
<td style="text-align: left;">쓰지 않기 (<code>__init__</code>에서 <code>self.리스트 = []</code>)</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 클래스 변수에 리스트
> - 딕셔너리를 두면 모든 객체가 공유
> - 객체마다 다른 값은 `__init__`에서 `self.`로

> ✅ **한 줄 정리** · 객체마다 다른 값은 `self.변수`, 모두 공유할 값은 클래스 변수로 두고 `클래스.변수`로 바꾼다.

**딥러닝 코드에서는**

- `self.items = items`
- 클래스 변수에 두면 train
- val 데이터셋이 같은 목록을 공유하는 사고가 남

---
## 3. 특수 메서드

> **이 장의 흐름**
>
> - 2장까지는 메서드를 이름으로 불렀다
> - `len()` · `[i]` · `print()` · `객체(x)` 같은 파이썬 기본 문법을 내가 만든 객체에 연결하는 법

**파이프라인 위치** · ② 불러오기 · ④ 모델 · ⑤ 학습 — `len(dataset)`, `dataset[i]`, `model(x)`

### 3-1. `B` 연산자 = 특수 메서드 · `isinstance()` · `__add__` <sub>원본 03 · 1~3절</sub>

> 🎯 **`10 + 5`나 `len(x)` 같은 기본 문법은 내부에서 무엇을 부르는가?**

💡 **쉽게 말하면**

- `+` · `len()` · `[ ]`은 **리모컨 버튼**, 특수 메서드(`__add__` · `__len__` · `__getitem__`)는 **버튼과 연결된 회로**.
- 버튼을 누르면 그 객체의 회로가 작동.

<img src="images/cls2_dunder.png" alt="기본 문법이 부르는 특수 메서드" width="763">

**실행 흐름**

1. int · list도 클래스
2. isinstance
3. + · * · bool
4. len · [ ] · in
5. 회로 없는 Car에 +

**먼저 예측** · `car1 + car2`(1-2의 Car)를 실행하면? — 답은 코드 실행 결과 아래에

In [7]:
# 기본 자료형도 클래스, 값은 그 클래스로 만든 객체
print(int, list)                                      # 자료형 이름 = 클래스
print(type(10).__name__, type([1, 2]).__name__)       # 값의 클래스 이름
print(isinstance(10, int), isinstance([1, 2], dict))  # 그 클래스의 객체인가


# 연산자 = 특수 메서드 호출
number = 10                             # int 객체
print(number + 5, number.__add__(5))    # a + b → a.__add__(b)
print(number * 3, number.__mul__(3))    # a * b → a.__mul__(b)
print(bool(number), number.__bool__())  # bool(a) → a.__bool__()


# len() · [ ] · in 도 특수 메서드
x = [10, 20, 30]                    # list 객체
print(len(x), x.__len__())          # len(x) → x.__len__()
print(x[1], x.__getitem__(1))       # x[i] → x.__getitem__(i)
print(20 in x, x.__contains__(20))  # a in x → x.__contains__(a)


# 특수 메서드가 없는 객체에 연산자를 쓰면
try:
    car1 + car2             # Car에는 __add__가 없음
except TypeError as e:      # + 를 지원하지 않음
    print("TypeError:", e)  # 에러 문장 출력

<class 'int'> <class 'list'>
int list
True False
15 15
30 30
True True
3 3
20 20
True True
TypeError: unsupported operand type(s) for +: 'Car' and 'Car'


**결과** · 예측 정답은 `TypeError` — Car에는 `+`와 연결된 `__add__`가 없음

- `<class 'int'> <class 'list'>` — int · list도 클래스
- `True False` — 10은 int의 객체, [1, 2]는 dict의 객체가 아님
- `15 15` · `30 30` · `3 3` · `20 20` — 연산자 · len · [ ]와 특수 메서드(`__add__` · `__mul__` · `__len__` · `__getitem__`) 결과 동일, bool · in도 True True
- `TypeError: unsupported operand type(s) for +: 'Car' and 'Car'` — `__add__`가 없으면 `+` 사용 불가

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">연산자 · 내장 함수 <code>a + b</code> · <code>len(x)</code></td>
<td style="text-align: left;">읽기 쉬움, 내부에서 특수 메서드 호출</td>
<td style="text-align: left;"><strong>항상</strong> (기본 선택)</td>
</tr>
<tr>
<td style="text-align: left;">특수 메서드 직접 <code>a.__add__(b)</code></td>
<td style="text-align: left;">같은 결과, 읽기 어려움</td>
<td style="text-align: left;">원리 확인용으로만</td>
</tr>
<tr>
<td style="text-align: left;"><code>type(x)</code></td>
<td style="text-align: left;">그 값의 <strong>클래스</strong> 자체</td>
<td style="text-align: left;">자료형 출력 · 디버깅</td>
</tr>
<tr>
<td style="text-align: left;"><code>isinstance(x, 클래스)</code></td>
<td style="text-align: left;">그 클래스의 객체인가 <strong>True/False</strong> (상속까지 인정, 4-1)</td>
<td style="text-align: left;">조건문에서 자료형 확인</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 특수 메서드를 직접 부르지 않음
> - `x.__len__()` 대신 `len(x)`처럼 문법으로

> ✅ **한 줄 정리** · 모든 값은 클래스의 객체이고, 연산자 · `len()` · `[ ]`은 그 객체의 특수 메서드를 부른다.

**딥러닝 코드에서는**

- `tensor_a + tensor_b` · `len(dataset)` · `dataset[0]` · 텐서
- 데이터셋이 특수 메서드를 정의했기 때문에 가능

### 3-2. `A` 출력 형태 정하기 · `__str__` · `__repr__` <sub>원본 03 · 4~8절 + 01</sub>

> 🎯 **`print(객체)`와 리스트 출력에서 원하는 글자가 나오게 하려면?**

💡 **쉽게 말하면**

- `__str__`은 **손님용 메뉴판**, `__repr__`은 **주방용 재료표**.
- 손님에게는 읽기 쉽게, 주방에는 다시 만들 수 있을 만큼 정확하게.

**실행 흐름**

1. 기본 출력(주소)
2. `__str__` · `__repr__` 정의
3. print · f-string · repr · 리스트 비교
4. `__repr__`만 있을 때

**먼저 예측** · `print([p1, p2])`는 `__str__`과 `__repr__` 중 어느 쪽을 쓸까? — 답은 코드 실행 결과 아래에

In [8]:
# 출력 방법을 정하지 않은 객체
class BasicProduct:                   # 출력 방법이 없는 클래스
    def __init__(self, name, price):  # 이름 · 가격 받기
        self.name = name              # 상품 이름
        self.price = price            # 가격


print(str(BasicProduct("Keyboard", 35000)).split(" at ")[0], "at 0x...>")  # 이름 + 주소뿐


# __str__ · __repr__ 정의
class Product:                        # 출력 방법을 정한 클래스
    def __init__(self, name, price):  # 이름 · 가격 받기
        self.name = name              # 상품 이름
        self.price = price            # 가격

    def __str__(self):                           # 사용자용: print() · str() · f-string
        return f"{self.name}: {self.price:,}원"  # 천 단위 쉼표로 읽기 쉽게

    def __repr__(self):  # 개발자용: repr() · 리스트 안 · 셀 마지막 줄
        return f"Product(name={self.name!r}, price={self.price})"  # !r → 따옴표까지 붙은 repr 값


p1 = Product("Keyboard", 35000)  # 상품 1
p2 = Product("Mouse", 18000)     # 상품 2


# 어디서 어느 쪽이 쓰이는지 확인
print(p1)             # __str__
print(f"주문: {p1}")  # __str__
print(repr(p1))       # __repr__
print([p1, p2])       # 리스트 안 → 원소마다 __repr__


# __repr__만 있으면 print도 __repr__을 사용
class OnlyRepr:              # __repr__만 있는 클래스
    def __repr__(self):      # __str__ 없음
        return "OnlyRepr()"  # 개발자용 표현


print(OnlyRepr(), [OnlyRepr()])  # print · 리스트 둘 다 __repr__

<__main__.BasicProduct object at 0x...>
Keyboard: 35,000원
주문: Keyboard: 35,000원
Product(name='Keyboard', price=35000)
[Product(name='Keyboard', price=35000), Product(name='Mouse', price=18000)]
OnlyRepr() [OnlyRepr()]


**결과** · 예측 정답은 `__repr__` — 리스트 안의 원소는 개발자용 표현으로 출력

- `<__main__.BasicProduct object at 0x...>` — 정의 전: 클래스 이름 + 주소뿐
- `Keyboard: 35,000원` · `주문: Keyboard: 35,000원` — print · f-string → `__str__`
- `Product(name='Keyboard', price=35000)` · `Product(name='Mouse', price=18000)]` — repr() · 리스트 안 → 원소마다 `__repr__` (`!r`로 따옴표 포함)
- `OnlyRepr() [OnlyRepr()]` — `__str__`이 없으면 print도 `__repr__` 사용

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>__str__</code></td>
<td style="text-align: left;"><strong>사용자용</strong> · 읽기 쉬운 문장</td>
<td style="text-align: left;"><code>print()</code> · <code>str()</code> · f-string 결과를 꾸밀 때</td>
</tr>
<tr>
<td style="text-align: left;"><code>__repr__</code></td>
<td style="text-align: left;"><strong>개발자용</strong> · <code>클래스(속성=값)</code> 형태</td>
<td style="text-align: left;">리스트 안 · 디버깅 · 셀 마지막 줄 출력</td>
</tr>
<tr>
<td style="text-align: left;"><code>{값}</code> vs <code>{값!r}</code></td>
<td style="text-align: left;"><code>str()</code> 결과 vs <code>repr()</code> 결과 (문자열에 따옴표)</td>
<td style="text-align: left;"><code>__repr__</code> 안의 문자열 속성은 <code>!r</code></td>
</tr>
<tr>
<td style="text-align: left;">둘 다 정의 안 함</td>
<td style="text-align: left;">object의 기본값 → <code>&lt;클래스 object at 0x...&gt;</code></td>
<td style="text-align: left;">출력할 일이 없는 클래스</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 리스트를 print할 때는 `__str__`이 아니라 `__repr__`이 쓰임
> - 하나만 만든다면 `__repr__`

> ✅ **한 줄 정리** · `__str__`은 사용자가 읽을 글자, `__repr__`은 개발자가 확인할 글자를 돌려주며, 하나만 만든다면 `__repr__`이다.

**딥러닝 코드에서는**

- `print(model)`에 층 구조가 나오는 것은 `nn.Module`이 `__repr__`을 정의했기 때문 (5장에서 확인)

### 3-3. `A` 길이와 번호로 꺼내기 · `__len__` · `__getitem__` <sub>원본 03 보충 · Dataset 원리</sub>

> 🎯 **내가 만든 객체에 `len()`과 `[i]`를 쓰려면 무엇을 정의해야 하는가?**

💡 **쉽게 말하면**

- `__len__`은 **'몇 개 있나요?'에 답하는 창구**, `__getitem__`은 **'3번을 달라'는 요청에 답하는 창구**.
- 물건을 미리 다 꺼내 두지 않고, 요청이 오면 그때 꺼내 줌.

<img src="images/cls2_dataset.png" alt="len과 번호로 꺼내는 Dataset 원리" width="779">

**실행 흐름**

1. SquareDataset 정의
2. len · ds[3]
3. `__dict__`로 저장 내용 확인
4. for 문
5. 범위 밖 IndexError
6. `__len__` 없는 Car

**먼저 예측** · `__iter__`를 만들지 않았는데 `for pair in ds`가 동작할까? — 답은 코드 실행 결과 아래에

In [9]:
# 번호를 주면 그때 계산해서 돌려주는 데이터셋
class SquareDataset:        # Dataset 원리를 보는 작은 클래스
    def __init__(self, n):  # n = 데이터 개수
        self.n = n          # 개수만 저장 (데이터를 미리 만들지 않음)

    def __len__(self):  # len(ds) → 전체 개수
        return self.n   # 저장해 둔 개수

    def __getitem__(self, i):                 # ds[i] → i번째 (입력, 정답) 한 쌍
        if not 0 <= i < self.n:               # 0 ~ n-1 밖이면
            raise IndexError("범위 밖 번호")  # for 문이 멈추는 신호
        return i, i * i                       # 요청받은 순간 계산


ds = SquareDataset(5)  # 5개짜리 데이터셋


# len() · [ ] 사용
print("len:", len(ds))            # __len__ 호출
print("ds[3]:", ds[3])            # __getitem__(3) 호출
print("저장된 값:", ds.__dict__)  # 결과 5개를 들고 있지 않음


# for 문: 0, 1, 2, ... 차례로 꺼내다 IndexError에서 멈춤
print("for:", [pair for pair in ds])  # 0번부터 차례로 꺼냄


# 범위 밖 번호
try:
    ds[5]                    # 5는 범위 밖
except IndexError as e:      # 범위 밖 번호
    print("IndexError:", e)  # 에러 문장 출력


# __len__이 없는 클래스에 len()을 쓰면
try:
    len(car1)               # Car에는 __len__이 없음
except TypeError as e:      # len()을 지원하지 않음
    print("TypeError:", e)  # 에러 문장 출력

len: 5
ds[3]: (3, 9)
저장된 값: {'n': 5}
for: [(0, 0), (1, 1), (2, 4), (3, 9), (4, 16)]
IndexError: 범위 밖 번호
TypeError: object of type 'Car' has no len()


**결과** · 예측 정답은 동작 — 0, 1, 2, …로 `__getitem__`을 부르다 `IndexError`가 나면 멈춤

- `len: 5` · `ds[3]: (3, 9)` — `len(ds)` → `__len__` · `ds[3]` → `__getitem__(3)` → (입력, 정답)
- `저장된 값: {'n': 5}` — 결과를 미리 만들지 않고 요청 때 계산
- `for: [(0, 0), (1, 1), (2, 4), (3, 9), (4, 16)]` — IndexError에서 반복 종료
- `IndexError: 범위 밖 번호` · `TypeError: object of type 'Car' has no len()` — 범위 밖 번호는 IndexError · `__len__`이 없으면 `len()` 불가

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>__getitem__</code>에서 그때 계산 · 읽기</td>
<td style="text-align: left;">메모리에 <strong>번호 · 경로만</strong> 보관</td>
<td style="text-align: left;">이미지처럼 전부 올리면 메모리가 부족할 때 (<strong>Dataset 기본</strong>)</td>
</tr>
<tr>
<td style="text-align: left;"><code>__init__</code>에서 전부 만들어 리스트에 저장</td>
<td style="text-align: left;">처음에 한 번 계산, 이후 빠름</td>
<td style="text-align: left;">작은 데이터 · 숫자 표</td>
</tr>
<tr>
<td style="text-align: left;"><code>__getitem__</code> (+ IndexError)</td>
<td style="text-align: left;"><strong>번호로</strong> 꺼내기, for 문도 자동</td>
<td style="text-align: left;">번호로 접근하는 데이터 (map-style Dataset)</td>
</tr>
<tr>
<td style="text-align: left;"><code>__iter__</code></td>
<td style="text-align: left;"><strong>순서대로만</strong> 꺼냄, 번호 접근 X</td>
<td style="text-align: left;">끝을 모르는 스트림 데이터 (IterableDataset, 드묾)</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 범위 밖 번호에서 `IndexError`를 내지 않으면 for 문이 끝나지 않음 (리스트에서 꺼내면 리스트가 대신 냄)

> ✅ **한 줄 정리** · `__len__`은 전체 개수, `__getitem__`은 i번째 한 쌍을 돌려주며, 이 둘이 PyTorch `Dataset`의 약속이다.

**딥러닝 코드에서는**

- `class MyDataset(Dataset):`의 `__len__` · `__getitem__`
- `DataLoader`가 `len(ds)`로 개수를 알고 `ds[i]`로 꺼내 배치로 묶음 (5장)

### 3-4. `A` 객체를 함수처럼 부르기 · `__call__` <sub>원본 03 보충 · model(x) 원리</sub>

> 🎯 **`model(x)`처럼 객체에 괄호를 붙여 부르면 무엇이 실행되는가?**

💡 **쉽게 말하면**

- `__call__`은 객체에 단 **실행 버튼**.
- 버튼(`객체(x)`)을 누르면 공통 처리를 한 뒤 실제 계산(`forward`)으로 넘어감.

<img src="images/cls2_call.png" alt="객체를 부르면 call을 거쳐 forward" width="763">

**실행 흐름**

1. Scale 정의
2. double(5) · double.forward(5) 비교
3. 거친 횟수
4. callable
5. 설정값이 다른 두 객체

**먼저 예측** · `double(5)`와 `double.forward(5)`를 한 번씩 부른 뒤 `double.calls`는? — 답은 코드 실행 결과 아래에

In [10]:
# 객체를 함수처럼 부를 수 있게 만들기
class Scale:                # 층(layer)을 흉내 낸 클래스
    def __init__(self, k):  # k = 곱할 값
        self.k = k          # 설정값 (층의 가중치 역할)
        self.calls = 0      # __call__을 거친 횟수

    def __call__(self, x):      # 객체(x) → 객체.__call__(x)
        self.calls += 1         # 공통 처리 (PyTorch는 여기서 hook 등을 처리)
        return self.forward(x)  # 실제 계산은 forward에 맡김

    def forward(self, x):  # 실제 계산
        return x * self.k  # 입력 × 설정값


double = Scale(2)  # 2배 하는 객체


# 두 가지 호출 비교
print("double(5) →", double(5))                  # __call__ → forward
print("double.forward(5) →", double.forward(5))  # forward만 실행
print("__call__을 거친 횟수:", double.calls)     # forward 직접 호출은 세지 않음


# 함수처럼 부를 수 있는 객체인가
print("callable:", callable(double), callable(car1))  # 괄호로 부를 수 있는지


# 객체마다 설정값을 기억
triple = Scale(3)                                # 3배 하는 객체
print("같은 입력 10 →", double(10), triple(10))  # 객체마다 다른 k

double(5) → 10
double.forward(5) → 10
__call__을 거친 횟수: 1
callable: True False
같은 입력 10 → 20 30


**결과** · 예측 정답은 `1` — `forward`를 직접 부르면 `__call__`의 처리를 건너뜀

- `double(5) → 10` · `double.forward(5) → 10` — 계산 결과는 같음
- `__call__을 거친 횟수: 1` — 직접 호출한 forward는 세지 않음
- `callable: True False` — `__call__`이 있어야 괄호로 부를 수 있음
- `같은 입력 10 → 20 30` — 객체마다 설정값(k)을 기억 = 층마다 다른 가중치

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">일반 함수 <code>scale(x, k)</code></td>
<td style="text-align: left;">설정값을 <strong>매번</strong> 인자로 전달</td>
<td style="text-align: left;">설정이 없는 단순 계산</td>
</tr>
<tr>
<td style="text-align: left;"><code>__call__</code> 객체 <code>double(x)</code></td>
<td style="text-align: left;">설정값(가중치)을 <strong>객체가 기억</strong></td>
<td style="text-align: left;">층 · 모델 · 손실 함수</td>
</tr>
<tr>
<td style="text-align: left;"><code>model(x)</code></td>
<td style="text-align: left;"><code>__call__</code> → 공통 처리 → <code>forward</code></td>
<td style="text-align: left;"><strong>항상</strong> (기본 선택)</td>
</tr>
<tr>
<td style="text-align: left;"><code>model.forward(x)</code></td>
<td style="text-align: left;">계산만, 공통 처리 <strong>건너뜀</strong></td>
<td style="text-align: left;">쓰지 않기</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - PyTorch 모델은 `model.forward(x)`가 아니라 `model(x)`로 호출
> - 직접 부르면 hook 등 `__call__`의 처리가 빠짐

> ✅ **한 줄 정리** · `객체(x)`는 `__call__`을 부르고, PyTorch는 그 안에서 `forward`를 부르므로 모델은 `model(x)`로 호출한다.

**딥러닝 코드에서는**

- `out = model(images)` · `loss = criterion(out, targets)` · 모델
- 손실 함수 모두 `__call__`이 있는 객체

---
## 4. 상속 · 다형성 · super

> **이 장의 흐름**
>
> - 3장에서 Dataset · 모델이 지켜야 할 약속을 봤다
> - PyTorch가 미리 만든 `Dataset` · `nn.Module`을 물려받아 필요한 부분만 바꾸는 법

**파이프라인 위치** · ② 불러오기 · ④ 모델 — `class MyDataset(Dataset)`, `class MyModel(nn.Module)`

### 4-1. `A` 물려받기와 덮어쓰기 · `class 자식(부모)` <sub>원본 04 · 1~3 · 7절</sub>

> 🎯 **부모 클래스의 기능을 물려받고, 일부만 바꾸려면?**

💡 **쉽게 말하면**

- 상속은 **부모 레시피를 물려받아 쓰는 것**.
- 그대로 쓰면 상속, 한 단계를 바꿔 쓰면 오버라이딩.
- 파이썬은 메서드를 자식 → 부모 순서로 찾음.

<img src="images/cls2_inherit.png" alt="상속 관계와 메서드를 찾는 순서" width="763">

**실행 흐름**

1. Animal 정의
2. BasicDog(pass) · Dog · Cat
3. 메서드 호출
4. 찾는 순서(`__mro__`)
5. isinstance · type 비교

**먼저 예측** · `isinstance(dog, Animal)`과 `type(dog) is Animal`의 결과는? — 답은 코드 실행 결과 아래에

In [11]:
# 부모 클래스
class Animal:                  # 부모 클래스
    def __init__(self, name):  # 모든 동물의 이름
        self.name = name       # 자식도 물려받는 속성

    def introduce(self):                        # 자식이 그대로 쓸 메서드
        return f"제 이름은 {self.name}입니다."  # 이름으로 소개 문장

    def sound(self):                    # 자식이 바꿀 메서드
        return "동물이 소리를 냅니다."  # 기본 소리


# 자식 1: 아무것도 쓰지 않고 전부 물려받기
class BasicDog(Animal):  # class 자식(부모):
    pass


# 자식 2 · 3: sound()만 다시 정의 (오버라이딩)
class Dog(Animal):     # Animal을 물려받음
    def sound(self):   # Animal의 sound를 덮어씀
        return "멍멍"  # Dog만의 소리


class Cat(Animal):     # Animal을 물려받음
    def sound(self):   # Animal의 sound를 덮어씀
        return "야옹"  # Cat만의 소리


basic, dog, cat = BasicDog("초코"), Dog("콩이"), Cat("나비")  # 자식 객체 셋
print(basic.introduce(), basic.sound())  # 둘 다 Animal 것
print(dog.introduce(), dog.sound())      # introduce는 Animal 것, sound는 Dog 것


# 메서드를 찾는 순서
print("찾는 순서:", [c.__name__ for c in Dog.__mro__])  # 자식 → 부모 → object


# 상속 관계 확인
print(isinstance(dog, Dog), isinstance(dog, Animal), isinstance(cat, Dog))  # 자식 객체 = 부모 객체
print(type(dog) is Animal, issubclass(Dog, Animal))  # type()은 정확히 그 클래스만

제 이름은 초코입니다. 동물이 소리를 냅니다.
제 이름은 콩이입니다. 멍멍
찾는 순서: ['Dog', 'Animal', 'object']
True True False
False True


**결과** · 예측 정답은 `True` · `False` — 자식 객체는 부모의 객체로도 인정되지만 `type()`은 정확히 `Dog`

- `제 이름은 초코입니다. 동물이 소리를 냅니다.` — BasicDog는 둘 다 Animal 것을 사용
- `제 이름은 콩이입니다. 멍멍` — introduce는 물려받고 sound는 Dog 것
- `찾는 순서: ['Dog', 'Animal', 'object']` — 맨 위는 object → 기본 `__repr__`(3-2)도 여기서 옴
- `True True False` · `False True` — Dog 객체는 Animal의 객체이기도 함 (Cat은 Dog 아님) · `type()`은 정확히 그 클래스만, `issubclass`는 클래스끼리 관계

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">그대로 상속 (<code>pass</code>)</td>
<td style="text-align: left;">부모 메서드를 <strong>그대로</strong> 사용</td>
<td style="text-align: left;">이름만 다르고 동작은 같을 때</td>
</tr>
<tr>
<td style="text-align: left;">오버라이딩</td>
<td style="text-align: left;">같은 이름으로 <strong>완전히 교체</strong></td>
<td style="text-align: left;">자식마다 동작이 달라야 할 때 (<code>forward</code>)</td>
</tr>
<tr>
<td style="text-align: left;">새 메서드 추가</td>
<td style="text-align: left;">부모에 없는 기능 <strong>추가</strong></td>
<td style="text-align: left;">그 자식에만 필요한 기능</td>
</tr>
<tr>
<td style="text-align: left;"><code>isinstance(x, 부모)</code></td>
<td style="text-align: left;">상속까지 인정 → True</td>
<td style="text-align: left;"><strong>기본 선택</strong> (자료형 확인)</td>
</tr>
<tr>
<td style="text-align: left;"><code>type(x) is 클래스</code></td>
<td style="text-align: left;"><strong>정확히</strong> 그 클래스만</td>
<td style="text-align: left;">자식을 제외해야 할 때 (드묾)</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 부모를 쓰지 않은 `class Car:`도 `class Car(object):`와 같음 (원본의 `class Hoho(object)`)
> - 모든 클래스는 object를 상속

> ✅ **한 줄 정리** · `class 자식(부모):`로 물려받고, 같은 이름의 메서드를 다시 정의하면 자식 것이 먼저 쓰인다.

**딥러닝 코드에서는**

- `class PetClassifier(nn.Module):` · `parameters()` · `to(device)`
- `train()`을 만들지 않아도 물려받아 사용

### 4-2. `A` 같은 호출 · 다른 동작 · `다형성` <sub>원본 04 · 4절</sub>

> 🎯 **종류가 다른 객체들을 같은 코드 한 줄로 다루려면?**

💡 **쉽게 말하면**

- 다형성은 **같은 '소리 내!' 명령에 동물마다 다른 소리**를 내는 것.
- 명령하는 쪽은 상대가 누구인지 묻지 않음.

**실행 흐름**

1. Animal · Dog · Cat 리스트
2. 같은 sound() 반복
3. if로 가르는 방식과 비교
4. Cow 추가

**먼저 예측** · `Cow`를 추가하면 반복문과 `sound_by_if` 중 고쳐야 하는 것은? — 답은 코드 실행 결과 아래에

In [12]:
# 종류가 다른 객체를 한 리스트에
animals = [Animal("동물"), Dog("초코"), Cat("나비")]  # 종류가 다른 객체 셋


# 같은 호출 sound() → 객체마다 다른 결과
for animal in animals:                       # 종류를 묻지 않고 반복
    print(animal.name, "→", animal.sound())  # 어떤 클래스인지 확인하지 않음


# 다형성 없이 쓰면: 종류마다 if로 갈라야 함
def sound_by_if(animal):            # 다형성 없이 쓰는 방식
    if type(animal) is Dog:         # 종류마다 직접 검사
        return "멍멍"               # Dog일 때
    elif type(animal) is Cat:       # 또 검사
        return "야옹"               # Cat일 때
    return "동물이 소리를 냅니다."  # 나머지 전부


assert [sound_by_if(a) for a in animals] == [a.sound() for a in animals]  # 지금은 결과 같음


# 새 종류 추가: 클래스만 추가하면 반복문은 그대로
class Cow(Animal):     # 새 종류 추가
    def sound(self):   # Cow만의 소리
        return "음메"  # 반복문은 그대로 둠


animals.append(Cow("누렁이"))                              # 목록에 추가만
print([a.sound() for a in animals])                        # 음메까지 나옴
print("if 방식은 Cow를 모름 →", sound_by_if(animals[-1]))  # 함수도 고쳐야 함

동물 → 동물이 소리를 냅니다.
초코 → 멍멍
나비 → 야옹
['동물이 소리를 냅니다.', '멍멍', '야옹', '음메']
if 방식은 Cow를 모름 → 동물이 소리를 냅니다.


**결과** · 예측 정답은 `sound_by_if`만 — 반복문은 그대로 `음메`까지 출력

- `동물 → 동물이 소리를 냅니다.` · `초코 → 멍멍` · `나비 → 야옹` — 같은 `sound()` 호출, 객체마다 다른 결과
- `['동물이 소리를 냅니다.', '멍멍', '야옹', '음메']` — Cow 클래스만 추가 → 반복문 수정 없음
- `if 방식은 Cow를 모름 → 동물이 소리를 냅니다.` — if 방식은 종류가 늘 때마다 함수도 고쳐야 함

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">다형성 <code>a.sound()</code></td>
<td style="text-align: left;">객체가 <strong>스스로</strong> 알맞은 메서드 실행</td>
<td style="text-align: left;"><strong>기본 선택</strong> — 종류가 늘어날 수 있을 때</td>
</tr>
<tr>
<td style="text-align: left;"><code>if type(a) is Dog:</code> 분기</td>
<td style="text-align: left;">부르는 쪽이 <strong>종류마다</strong> 갈라야 함, 종류 추가 시 함수도 수정</td>
<td style="text-align: left;">종류가 고정된 짧은 코드</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 자식마다 메서드 이름
> - 인자를 똑같이 맞춰야 함
> - `sound()`와 `make_sound()`처럼 다르면 다형성이 깨짐

> ✅ **한 줄 정리** · 같은 이름의 메서드를 자식마다 다르게 정의하면, 종류를 묻지 않고 같은 호출로 다룰 수 있다.

**딥러닝 코드에서는**

- `for layer in layers: x = layer(x)` · `nn.Sequential`이 Conv · ReLU
- Linear를 종류 구분 없이 차례로 부르는 원리

### 4-3. `A` 부모 기능 확장 · `super()` <sub>원본 04 · 5~6절</sub>

> 🎯 **부모 기능을 버리지 않고 그 위에 덧붙이려면?**

💡 **쉽게 말하면**

- `super()`는 **'부모 방식대로 먼저 하고'**.
- 그다음 자식이 필요한 것만 덧붙임.
- 생성자에서 이를 빠뜨리면 부모가 준비할 값(`name`)이 비어 있음.

<img src="images/cls2_super.png" alt="super로 부모 초기화 뒤 자식 속성 추가" width="763">

**실행 흐름**

1. LoudDog: 부모 sound + !!!
2. Bird: `super().__init__` + can_fly
3. `__dict__` 확인
4. BrokenBird: super 생략 AttributeError

**먼저 예측** · `super().__init__(name)`을 빠뜨린 `BrokenBird("펭귄", False)`는 만드는 순간 에러가 날까? — 답은 코드 실행 결과 아래에

In [13]:
# 부모 메서드 결과에 덧붙이기
class LoudDog(Dog):                     # Dog를 물려받음
    def sound(self):                    # 부모 sound를 확장
        return super().sound() + "!!!"  # super() → 부모(Dog)의 sound()


print(LoudDog("두부").sound())  # 멍멍 + !!!


# 생성자 확장: 부모 초기화 + 자식 속성 추가
class Bird(Animal):                     # Animal을 물려받음
    def __init__(self, name, can_fly):  # 자식은 인자가 하나 더
        super().__init__(name)          # name 저장은 부모(Animal)에게
        self.can_fly = can_fly          # Bird에만 필요한 속성

    def sound(self):   # Animal의 sound를 덮어씀
        return "짹짹"  # Bird만의 소리

    def show_flight(self):  # 자식에만 있는 새 메서드
        # can_fly로 문장 고르기
        return f"{self.name}: 날 수 {'있습니다' if self.can_fly else '없습니다'}."


bird = Bird("참새", True)                                  # name · can_fly 둘 다 전달
print(bird.introduce(), bird.sound(), bird.show_flight())  # 물려받은 · 바꾼 · 새 메서드
print("bird에 저장된 값:", bird.__dict__)                  # name도 들어 있는지


# super().__init__()을 빠뜨리면
class BrokenBird(Animal):               # super()를 빠뜨린 자식
    def __init__(self, name, can_fly):  # 부모 초기화를 빠뜨림
        self.can_fly = can_fly          # 부모 초기화 생략 → name이 저장되지 않음


try:
    BrokenBird("펭귄", False).introduce()  # 에러는 생성이 아니라 name을 쓰는 순간
except AttributeError as e:                # name 속성이 없음
    print("AttributeError:", e)            # 에러 문장 출력

멍멍!!!
제 이름은 참새입니다. 짹짹 참새: 날 수 있습니다.
bird에 저장된 값: {'name': '참새', 'can_fly': True}
AttributeError: 'BrokenBird' object has no attribute 'name'


**결과** · 예측 정답은 아니요 — 만들어지지만 `name`이 없어서 `introduce()`를 부를 때 `AttributeError`

- `멍멍!!!` — 부모(Dog) 결과 + 덧붙임
- `제 이름은 참새입니다. 짹짹 참새: 날 수 있습니다.` — introduce는 Animal 것, sound · show_flight는 Bird 것
- `bird에 저장된 값: {'name': '참새', 'can_fly': True}` — name은 부모가, can_fly는 자식이 저장
- `AttributeError: 'BrokenBird' object has no attribute 'name'` — 부모 초기화를 건너뛰어 name이 없음

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">오버라이딩만</td>
<td style="text-align: left;">부모 메서드를 <strong>완전히 교체</strong></td>
<td style="text-align: left;">부모 동작이 전혀 필요 없을 때</td>
</tr>
<tr>
<td style="text-align: left;"><code>super().메서드()</code></td>
<td style="text-align: left;">부모 결과를 받아 <strong>덧붙임</strong></td>
<td style="text-align: left;">부모 기능 + 추가 (<code>LoudDog</code>)</td>
</tr>
<tr>
<td style="text-align: left;"><code>super().__init__(...)</code></td>
<td style="text-align: left;">부모 속성 준비 <strong>후</strong> 자식 속성 추가</td>
<td style="text-align: left;">자식에 <code>__init__</code>이 있으면 <strong>항상</strong></td>
</tr>
<tr>
<td style="text-align: left;"><code>부모.__init__(self, ...)</code></td>
<td style="text-align: left;">부모 이름을 직접 씀 (2-1 방식)</td>
<td style="text-align: left;">옛 코드에서만 · 부모가 바뀌면 함께 고쳐야 함</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 자식에 `__init__`을 만들면 부모 `__init__`은 자동 실행 X · 첫 줄에 `super().__init__(...)`

> ✅ **한 줄 정리** · `super().메서드()`로 부모 기능을 먼저 실행하고, 자식의 `__init__` 첫 줄에는 `super().__init__()`을 둔다.

**딥러닝 코드에서는**

- `super().__init__()` 다음 줄에 `self.fc = nn.Linear(...)`
- nn.Module은 빠뜨리면 층을 저장하는 순간 바로 에러 (5장)

---
## 5. 종합 실습 · 직접 만드는 Dataset · 모델 → PyTorch

> **이 장의 흐름**
>
> - 1~4장 문법을 PyTorch에서 그대로 확인
> - 01_Python문법 8장에서 만든 `(경로, 번호)` 목록을 받는 Dataset과, 그 배치를 받는 모델을 만듦

<img src="images/cls2_torch.png" alt="목록 → Dataset → DataLoader → 모델" width="763">

**파이프라인 위치** · ② 불러오기 → ④ 모델 → ⑤ 학습 직전

> 🎯 **PyTorch `Dataset` · `nn.Module`은 1~4장의 어떤 문법으로 만들어지는가?**

💡 **쉽게 말하면**

- PyTorch는 **빈칸이 있는 설계도**를 줌.
- `Dataset`은 `__len__` · `__getitem__` 칸을, `nn.Module`은 `__init__` · `forward` 칸을 채움.
- 나머지(배치 묶기 · 가중치 관리 · `model(x)`)는 물려받은 코드가 처리.

**실행 흐름**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">단계</th>
<th style="text-align: left;">하는 일</th>
<th style="text-align: left;">쓰는 문법</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;"><code>(경로, 번호)</code> 목록 만들기</td>
<td style="text-align: left;">01_Python문법 8장 · 컴프리헨션</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;"><code>PetDataset(Dataset)</code> · <code>__len__</code> · <code>__getitem__</code></td>
<td style="text-align: left;">4-1 상속 · 3-3 · 2-2 <code>self.items</code></td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;"><code>DataLoader</code>로 배치 묶기</td>
<td style="text-align: left;">3-3 <code>len(ds)</code> · <code>ds[i]</code></td>
</tr>
<tr>
<td style="text-align: left;">4</td>
<td style="text-align: left;"><code>PetClassifier(nn.Module)</code> · <code>super().__init__()</code> · <code>forward</code></td>
<td style="text-align: left;">4-3 · 3-4 · 3-2 <code>print(model)</code></td>
</tr>
<tr>
<td style="text-align: left;">5</td>
<td style="text-align: left;"><code>model(x)</code> → forward · 파라미터 수</td>
<td style="text-align: left;">3-4 <code>__call__</code> · 4-1 물려받은 <code>parameters()</code></td>
</tr>
<tr>
<td style="text-align: left;">6</td>
<td style="text-align: left;">종류가 다른 층을 같은 호출로 차례로</td>
<td style="text-align: left;">4-2 다형성</td>
</tr>
<tr>
<td style="text-align: left;">7</td>
<td style="text-align: left;"><code>super().__init__()</code> 생략 에러</td>
<td style="text-align: left;">4-3</td>
</tr>
</tbody>
</table>

**먼저 예측** · `super().__init__()`을 빠뜨린 `BrokenClassifier()`는 4-3의 `BrokenBird`처럼 나중에 에러가 날까, 만드는 순간 에러가 날까? — 답은 코드 실행 결과 아래에

In [14]:
import torch                                      # 텐서
from torch import nn                              # 층 · 모델
from torch.utils.data import Dataset, DataLoader  # 데이터셋 부모 · 배치 공급


# 1단계: 01_Python문법 8장과 같은 형태의 목록 [(경로, 번호), ...]
counts = {"bird": 3, "cat": 5, "dog": 4}                           # 클래스별 가짜 이미지 수
class_to_idx = {name: i for i, name in enumerate(sorted(counts))}  # 이름순 번호
items = [(f"pets/{name}/img_{i:03d}.jpg", class_to_idx[name])      # (경로, 번호) 쌍
         for name, n in counts.items() for i in range(n)]          # 클래스마다 n장
print("항목 수:", len(items), "/ 첫 항목:", items[0])              # 목록 확인


# 2단계: Dataset 상속 + __len__ · __getitem__
class PetDataset(Dataset):              # 4-1 상속
    def __init__(self, items, size=8):  # 목록 · 이미지 크기
        self.items = items              # 2-2 인스턴스 변수 (목록만 저장)
        self.size = size                # 가짜 이미지 한 변

    def __len__(self):          # 3-3 len(ds)
        return len(self.items)  # 목록 길이 = 데이터 수

    def __getitem__(self, i):                                         # 3-3 ds[i]
        path, target = self.items[i]                                  # i번째 경로 · 번호
        image = torch.full((3, self.size, self.size), float(target))  # 가짜 이미지 (값 = 번호)
        return image, target                                          # 1-1 (입력, 정답) 한 쌍


ds = PetDataset(items)                                          # 데이터셋 객체
image, target = ds[0]                                           # __getitem__(0)
print("len:", len(ds), "/ ds[0]:", tuple(image.shape), target)  # 샘플 1개의 모양
print("Dataset의 객체?", isinstance(ds, Dataset))               # 상속 확인


# 3단계: DataLoader가 len() · ds[i]로 꺼내 배치로 묶음
loader = DataLoader(ds, batch_size=4, shuffle=False)       # 4개씩 묶기 (순서 그대로)
print("배치 수:", len(loader))                             # 12 ÷ 4
for images, targets in loader:                             # ds[0]~ds[3] → 1번째 배치, ...
    print("배치:", tuple(images.shape), targets.tolist())  # 묶인 모양 · 번호

항목 수: 12 / 첫 항목: ('pets/bird/img_000.jpg', 0)
len: 12 / ds[0]: (3, 8, 8) 0
Dataset의 객체? True
배치 수: 3
배치: (4, 3, 8, 8) [0, 0, 0, 1]
배치: (4, 3, 8, 8) [1, 1, 1, 1]
배치: (4, 3, 8, 8) [2, 2, 2, 2]


In [15]:
# 4단계: nn.Module 상속 + super().__init__() + forward
class PetClassifier(nn.Module):                            # nn.Module을 물려받은 모델
    def __init__(self, num_classes=3, size=8):             # 클래스 수 · 이미지 크기
        super().__init__()                                 # 4-3 nn.Module 준비를 먼저
        self.flatten = nn.Flatten()                        # (배치, 3, 8, 8) → (배치, 192)
        self.fc = nn.Linear(3 * size * size, num_classes)  # 192 → 3

    def forward(self, x):                # 3-4 model(x)가 부르는 계산
        return self.fc(self.flatten(x))  # 펴기 → 점수 3개


torch.manual_seed(0)     # 가중치 초기값 고정
model = PetClassifier()  # 모델 객체
print(model)             # 3-2 nn.Module이 정의한 __repr__


# 5단계: model(x) → __call__ → forward
out = model(images)                                                   # 마지막 배치 4장
print("출력 모양:", tuple(out.shape))                                 # (배치, 클래스 수)
print("forward와 같은 값:", torch.equal(out, model.forward(images)))  # 계산 결과는 같음
print("파라미터 수:", sum(p.numel() for p in model.parameters()))     # 192 × 3 + 3


# 6단계: 층마다 계산이 달라도 같은 호출 layer(x)
x = images                                                  # 첫 층에 넣을 입력
for layer in model.children():                              # 4-2 다형성: Flatten → Linear
    x = layer(x)                                            # 층마다 같은 호출
    print(f"{type(layer).__name__:>7} → {tuple(x.shape)}")  # 층 이름 · 나온 모양


# 7단계: super().__init__()을 빠뜨리면
class BrokenClassifier(nn.Module):   # super()를 빠뜨린 모델
    def __init__(self):              # super().__init__() 없음
        self.fc = nn.Linear(192, 3)  # nn.Module 준비 전에 층을 저장


try:
    BrokenClassifier()           # 만드는 순간 실행
except AttributeError as e:      # 모듈 등록 공간이 없음
    print("AttributeError:", e)  # 에러 문장 출력

PetClassifier(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc): Linear(in_features=192, out_features=3, bias=True)
)
출력 모양: (4, 3)
forward와 같은 값: True
파라미터 수: 579
Flatten → (4, 192)
 Linear → (4, 3)
AttributeError: cannot assign module before Module.__init__() call


**결과** · 예측 정답은 만드는 순간 — `self.fc = nn.Linear(...)`로 층을 저장할 때 nn.Module이 준비되지 않았다고 바로 알려 줌 (4-3의 `BrokenBird`는 `name`을 쓸 때 에러)

- `항목 수: 12 / 첫 항목: ('pets/bird/img_000.jpg', 0)` · `len: 12 / ds[0]: (3, 8, 8) 0` · `Dataset의 객체? True` · `배치 수: 3` · `배치: (4, 3, 8, 8) [0, 0, 0, 1]` — 샘플 1개는 (3채널 8×8, 번호) · 12 ÷ 4 = 3배치, 묶을 때 맨 앞에 배치 차원 4
- `(fc): Linear(in_features=192, out_features=3, bias=True)` · `파라미터 수: 579` — `print(model)`은 nn.Module의 `__repr__` · 192 × 3 + 3 = 579
- `출력 모양: (4, 3)` · `forward와 같은 값: True` · `Flatten → (4, 192)` · `Linear → (4, 3)` — `model(x)` = `__call__` → forward, 층마다 같은 `layer(x)`
- `AttributeError: cannot assign module before Module.__init__() call` — nn.Module은 super 생략을 층 저장 순간 바로 알려 줌

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">직접 만든 <code>SquareDataset</code> (3-3)</td>
<td style="text-align: left;"><code>__len__</code> · <code>__getitem__</code>만, 배치 묶기는 직접</td>
<td style="text-align: left;">원리 확인 · 아주 작은 데이터</td>
</tr>
<tr>
<td style="text-align: left;"><code>Dataset</code> 상속 <code>PetDataset</code></td>
<td style="text-align: left;">같은 약속 + <code>DataLoader</code> 등 PyTorch 도구와 호환</td>
<td style="text-align: left;"><strong>기본 선택</strong> (모든 학습 코드)</td>
</tr>
<tr>
<td style="text-align: left;">직접 만든 <code>Scale</code> (3-4)</td>
<td style="text-align: left;"><code>__call__</code> → forward만</td>
<td style="text-align: left;">원리 확인</td>
</tr>
<tr>
<td style="text-align: left;"><code>nn.Module</code> 상속 <code>PetClassifier</code></td>
<td style="text-align: left;"><code>__call__</code> · <code>parameters()</code> · <code>to(device)</code> · <code>__repr__</code> 물려받음</td>
<td style="text-align: left;"><strong>기본 선택</strong> (모든 모델)</td>
</tr>
<tr>
<td style="text-align: left;"><code>nn.Sequential(...)</code></td>
<td style="text-align: left;">forward 없이 층을 순서대로 호출</td>
<td style="text-align: left;">층을 일렬로 쌓기만 할 때</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - `__getitem__`은 샘플 1개 기준 모양 `(3, 8, 8)`을 돌려주고 배치 차원 `4`는 `DataLoader`가 붙임
> - 직접 붙이면 `(4, 1, 3, 8, 8)`처럼 틀어짐

> ✅ **한 줄 정리** · `Dataset`은 `__len__` · `__getitem__`을, `nn.Module`은 `super().__init__()` · `forward`를 채운 상속 클래스이고, 모델은 `model(x)`로 부른다.

**딥러닝 코드에서는**

- 다음 단계 06_딥러닝기초·PyTorch의 학습 루프가 이 두 객체를 그대로 사용 (아래)

```python
for images, targets in loader:          # Dataset · DataLoader (3-3)
    out = model(images)                 # __call__ → forward (3-4)
    loss = criterion(out, targets)      # 손실 함수도 __call__ 객체
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()                    # 메서드로 가중치 상태 변경 (1-3)
```

---
## 그래서 오늘 무엇을 배웠는가?

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">학습 목표</th>
<th style="text-align: left;">할 수 있게 된 것</th>
<th style="text-align: left;">확인 위치</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1. 클래스 · self · 변수</td>
<td style="text-align: left;"><code>car1.price</code>는 객체마다, <code>Car.total_created</code>는 공유 · <code>a.total_created = 100</code>은 새 인스턴스 변수</td>
<td style="text-align: left;">1-2 · 2-1 · 2-2</td>
</tr>
<tr>
<td style="text-align: left;">2. 특수 메서드</td>
<td style="text-align: left;"><code>len(ds)</code> → <code>__len__</code>, <code>ds[3]</code> → <code>(3, 9)</code>, 리스트 안 출력은 <code>__repr__</code></td>
<td style="text-align: left;">3-1 ~ 3-4</td>
</tr>
<tr>
<td style="text-align: left;">3. 상속 · super</td>
<td style="text-align: left;"><code>Dog</code> → <code>Animal</code> → <code>object</code> 순서로 찾고, <code>super().__init__(name)</code>으로 부모가 name 저장</td>
<td style="text-align: left;">4-1 · 4-3</td>
</tr>
<tr>
<td style="text-align: left;">4. Dataset</td>
<td style="text-align: left;">12장 → <code>DataLoader(batch_size=4)</code> → 배치 3개, 모양 <code>(4, 3, 8, 8)</code></td>
<td style="text-align: left;">5</td>
</tr>
<tr>
<td style="text-align: left;">5. nn.Module</td>
<td style="text-align: left;"><code>model(images)</code> → <code>(4, 3)</code> · 파라미터 579개 · super 생략 시 즉시 에러</td>
<td style="text-align: left;">5</td>
</tr>
</tbody>
</table>

### 주제별 한 줄 · 반드시 기억할 것

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: center;">등급</th>
<th style="text-align: left;">주제</th>
<th style="text-align: left;">반드시 기억할 것</th>
<th style="text-align: left;">딥러닝 코드에서</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">클래스 · 객체</td>
<td style="text-align: left;"><code>class</code> + <code>__init__</code>, <code>클래스(...)</code>로 생성</td>
<td style="text-align: left;"><code>model = PetClassifier()</code></td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">메서드</td>
<td style="text-align: left;">그 객체의 값만 변경, 검증은 <code>raise</code></td>
<td style="text-align: left;"><code>optimizer.step()</code></td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">self</td>
<td style="text-align: left;">호출한 객체, 기억할 값은 <code>self.</code>에</td>
<td style="text-align: left;"><code>self.fc</code>, <code>self.items</code></td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;"><code>__str__</code> · <code>__repr__</code></td>
<td style="text-align: left;">사용자용 · 개발자용(리스트 안)</td>
<td style="text-align: left;"><code>print(model)</code></td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;"><code>__len__</code> · <code>__getitem__</code></td>
<td style="text-align: left;">전체 개수 · i번째 한 쌍</td>
<td style="text-align: left;"><code>Dataset</code> · <code>DataLoader</code></td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;"><code>__call__</code></td>
<td style="text-align: left;"><code>객체(x)</code> → forward</td>
<td style="text-align: left;"><code>model(x)</code>, <code>criterion(out, y)</code></td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">상속 · 오버라이딩</td>
<td style="text-align: left;"><code>class 자식(부모)</code>, 같은 이름 재정의</td>
<td style="text-align: left;"><code>class Net(nn.Module)</code></td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">다형성</td>
<td style="text-align: left;">같은 호출, 객체마다 다른 동작</td>
<td style="text-align: left;"><code>nn.Sequential</code></td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">super</td>
<td style="text-align: left;">부모 기능 + 추가, <code>__init__</code> 첫 줄</td>
<td style="text-align: left;"><code>super().__init__()</code></td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">종합</td>
<td style="text-align: left;">Dataset · nn.Module 상속</td>
<td style="text-align: left;">모든 학습 코드</td>
</tr>
<tr>
<td style="text-align: center;"><code>B</code></td>
<td style="text-align: left;">흩어진 데이터</td>
<td style="text-align: left;">관련 값은 한 덩어리로</td>
<td style="text-align: left;"><code>(image, label)</code> 한 쌍</td>
</tr>
<tr>
<td style="text-align: center;"><code>B</code></td>
<td style="text-align: left;">클래스 변수</td>
<td style="text-align: left;">공유 값은 <code>클래스.변수</code>, 리스트는 <code>self.</code>로</td>
<td style="text-align: left;">데이터셋마다 다른 목록</td>
</tr>
<tr>
<td style="text-align: center;"><code>B</code></td>
<td style="text-align: left;">특수 메서드</td>
<td style="text-align: left;">연산자 · <code>len</code> · <code>[ ]</code> → <code>__add__</code> 등</td>
<td style="text-align: left;"><code>tensor + tensor</code></td>
</tr>
</tbody>
</table>

---
## 기억 확인 문제

위 내용을 **보지 않고** 답한 뒤, ✅ 한 줄 정리와 비교해 ○ (정확) · △ (일부) · × (모름)를 표시. △ · ×인 섹션만 다시 보기.

### 1. 질문에 답하기

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">섹션</th>
<th style="text-align: left;">🎯 질문</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1-1</td>
<td style="text-align: left;">데이터를 리스트 두 개로 나눠 두면 어떤 문제가 생기는가?</td>
</tr>
<tr>
<td style="text-align: left;">1-2</td>
<td style="text-align: left;"><code>클래스(...)</code>로 객체를 만들 때 자동으로 실행되는 메서드와 그 역할은?</td>
</tr>
<tr>
<td style="text-align: left;">2-1</td>
<td style="text-align: left;"><code>self</code>는 무엇이며, 호출할 때는 왜 넣지 않는가?</td>
</tr>
<tr>
<td style="text-align: left;">2-2</td>
<td style="text-align: left;">인스턴스 변수와 클래스 변수의 차이는?</td>
</tr>
<tr>
<td style="text-align: left;">3-1</td>
<td style="text-align: left;"><code>a + b</code> · <code>len(x)</code>는 내부에서 무엇을 부르는가?</td>
</tr>
<tr>
<td style="text-align: left;">3-2</td>
<td style="text-align: left;"><code>__str__</code>과 <code>__repr__</code>은 각각 언제 쓰이는가?</td>
</tr>
<tr>
<td style="text-align: left;">3-3</td>
<td style="text-align: left;"><code>len(ds)</code> · <code>ds[i]</code>를 쓰려면 무엇을 정의하는가?</td>
</tr>
<tr>
<td style="text-align: left;">3-4</td>
<td style="text-align: left;"><code>model(x)</code>를 부르면 무엇이 실행되며, <code>forward</code>를 직접 부르면 안 되는 이유는?</td>
</tr>
<tr>
<td style="text-align: left;">4-1</td>
<td style="text-align: left;">상속과 오버라이딩의 차이는?</td>
</tr>
<tr>
<td style="text-align: left;">4-2</td>
<td style="text-align: left;">다형성이란?</td>
</tr>
<tr>
<td style="text-align: left;">4-3</td>
<td style="text-align: left;">자식 <code>__init__</code>에서 <code>super().__init__()</code>을 빠뜨리면?</td>
</tr>
<tr>
<td style="text-align: left;">5</td>
<td style="text-align: left;">PyTorch Dataset · 모델 클래스에 꼭 채워야 하는 메서드는?</td>
</tr>
</tbody>
</table>

#### 내 답 · 이 셀을 더블클릭해서 작성

**1-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**1-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-4** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**4-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**4-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**4-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**5** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

### 2. 코드로 답하기 · 연습 문제

- 수업 활동판을 정리한 문제
- `raise NotImplementedError(...)` 줄을 지우고 코드를 작성한 뒤 실행 → `통과`가 나오면 성공
- **테스트 부분은 수정하지 않기**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">연습</th>
<th style="text-align: left;">재료</th>
<th style="text-align: left;">확인하는 섹션</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;">활동 1-1 · 메뉴 <code>MenuItem</code></td>
<td style="text-align: left;">1-2 · 1-3 · 3-2</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;">활동 2-1 · 학생 <code>Student</code></td>
<td style="text-align: left;">2-2</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">활동 4-1 · 책 <code>Book</code></td>
<td style="text-align: left;">3-2</td>
</tr>
<tr>
<td style="text-align: left;">4</td>
<td style="text-align: left;">활동 8-1 · 게임 캐릭터 <code>Character</code></td>
<td style="text-align: left;">4-1 · 4-2 · 4-3</td>
</tr>
<tr>
<td style="text-align: left;">5</td>
<td style="text-align: left;">딥러닝 연결 · 문장 데이터셋 <code>TextDataset</code></td>
<td style="text-align: left;">3-3</td>
</tr>
</tbody>
</table>

In [16]:
# 연습 1 · 활동 1-1: TODO만 고친 뒤 실행 → "통과"가 나오면 성공
class MenuItem:                                                         # 메뉴 한 개
    def __init__(self, name, price):                                    # 이름 · 가격 받기
        raise NotImplementedError("TODO 1: 이름과 가격을 객체에 저장")  # 이 줄을 지우고 작성

    def apply_discount(self, rate):  # 할인율 받기
        # 이 줄을 지우고 작성
        raise NotImplementedError("TODO 2: 0~1 밖이면 ValueError, 아니면 가격을 정수로 변경")

    def __str__(self):  # print(메뉴)에 쓰임
        # 이 줄을 지우고 작성
        raise NotImplementedError("TODO 3: '아메리카노: 4,000원' 형식으로 반환")


# 테스트 (수정하지 않기)
try:
    coffee, tea = MenuItem("아메리카노", 5000), MenuItem("녹차", 4500)     # 메뉴 두 개
    assert str(coffee) == "아메리카노: 5,000원", "__str__ 형식 확인"       # __str__ 검사
    coffee.apply_discount(0.2)                                             # 20% 할인
    assert coffee.price == 4000 and tea.price == 4500, "할인은 그 객체만"  # coffee만 바뀌었는지
    try:
        coffee.apply_discount(1.5)                                         # 범위 밖 할인율
        raise AssertionError("범위 밖 할인율은 ValueError")                # 에러가 안 나면 실패
    except ValueError:                                                     # 에러가 나야 정상
        pass
except (AssertionError, NotImplementedError) as e:                         # 실패 · 미작성
    print("미완료 · 확인할 내용:", e)                                      # 무엇이 틀렸는지
else:
    print("연습 1 통과")                                                   # 모든 검사 통과

미완료 · 확인할 내용: TODO 1: 이름과 가격을 객체에 저장


In [17]:
# 연습 2 · 활동 2-1
class Student:            # 학생 한 명
    total_created = None  # TODO 1: 모든 학생이 공유할 생성 수를 0으로

    def __init__(self, name, score):  # 이름 · 점수 받기
        # 이 줄을 지우고 작성
        raise NotImplementedError("TODO 2: 이름 · 점수 저장 + 생성 수 1 증가")

    def update_score(self, score):                                  # 새 점수 받기
        raise NotImplementedError("TODO 3: 이 학생의 점수만 변경")  # 이 줄을 지우고 작성


# 테스트 (수정하지 않기)
try:
    s1, s2 = Student("민수", 80), Student("지수", 90)             # 학생 두 명
    assert Student.total_created == 2, "생성 수는 클래스 변수로"  # 공유 값 검사
    s1.update_score(85)                                           # s1 점수만 변경
    assert (s1.score, s2.score) == (85, 90), "점수는 그 학생만"   # s2는 그대로인지
    s1.total_created = 100                                        # 객체 이름으로 대입
    # 클래스 변수는 그대로인지
    assert Student.total_created == 2 and s2.total_created == 2, "대입은 클래스 변수 그대로"
except (AssertionError, NotImplementedError, TypeError) as e:     # 실패 · 미작성
    print("미완료 · 확인할 내용:", e)                             # 무엇이 틀렸는지
else:
    print("연습 2 통과")                                          # 모든 검사 통과

미완료 · 확인할 내용: TODO 2: 이름 · 점수 저장 + 생성 수 1 증가


In [18]:
# 연습 3 · 활동 4-1
class Book:                                    # 책 한 권
    def __init__(self, title, author, price):  # 제목 · 저자 · 가격
        self.title = title                     # 제목
        self.author = author                   # 저자
        self.price = price                     # 가격

    def __str__(self):  # 사용자용
        # 이 줄을 지우고 작성
        raise NotImplementedError("TODO 1: '파이썬 첫걸음 - 김코딩 / 20,000원'")

    def __repr__(self):  # 개발자용
        # 이 줄을 지우고 작성
        raise NotImplementedError("TODO 2: 개발자용 Book(title='...', author='...', price=...)")


# 테스트 (수정하지 않기)
try:
    # 책 두 권
    book1, book2 = Book("파이썬 첫걸음", "김코딩", 20000), Book("딥러닝 입문", "이파이", 25000)
    assert str(book1) == "파이썬 첫걸음 - 김코딩 / 20,000원", "__str__ 형식 확인"  # __str__ 검사
    expected = "Book(title='파이썬 첫걸음', author='김코딩', price=20000)"  # 기대하는 __repr__
    assert repr(book1) == expected, "__repr__ 형식 확인"                    # __repr__ 검사
    # 리스트 출력 검사
    assert str([book1, book2]).startswith("[Book(title="), "리스트 안에서는 __repr__"
except (AssertionError, NotImplementedError) as e:                          # 실패 · 미작성
    print("미완료 · 확인할 내용:", e)                                       # 무엇이 틀렸는지
else:
    print("연습 3 통과")                                                    # 모든 검사 통과

미완료 · 확인할 내용: TODO 1: '파이썬 첫걸음 - 김코딩 / 20,000원'


In [19]:
# 연습 4 · 활동 8-1
class Character:               # 부모 (완성됨)
    def __init__(self, name):  # 캐릭터 이름
        self.name = name       # 자식도 물려받음

    def attack(self):                     # 기본 공격
        return f"{self.name}: 기본 공격"  # 부모의 공격 문장


class Warrior(Character):                                     # Character를 물려받음
    def attack(self):                                         # Warrior의 공격
        raise NotImplementedError("TODO 1: '레온: 검 공격'")  # 이 줄을 지우고 작성


class Mage(Character):                                          # Character를 물려받음
    def attack(self):                                           # Mage의 공격
        raise NotImplementedError("TODO 2: '루나: 마법 공격'")  # 이 줄을 지우고 작성


class DoubleAttackWarrior(Warrior):  # Warrior를 물려받음
    def attack(self):                # 부모(Warrior) 공격 + 추가
        # 이 줄을 지우고 작성
        raise NotImplementedError("TODO 3: super()로 '카인: 검 공격 + 추가 공격'")


# 테스트 (수정하지 않기)
try:
    # 종류가 다른 넷
    team = [Character("로키"), Warrior("레온"), Mage("루나"), DoubleAttackWarrior("카인")]
    # 기대하는 공격 문장
    expected = ["로키: 기본 공격", "레온: 검 공격", "루나: 마법 공격", "카인: 검 공격 + 추가 공격"]
    assert [c.attack() for c in team] == expected, "attack() 결과 확인"  # 다형성 검사
    assert isinstance(team[3], Character), "DoubleAttackWarrior도 Character의 객체"  # 상속 검사
except (AssertionError, NotImplementedError) as e:                       # 실패 · 미작성
    print("미완료 · 확인할 내용:", e)                                    # 무엇이 틀렸는지
else:
    print("연습 4 통과")                                                 # 모든 검사 통과

미완료 · 확인할 내용: TODO 1: '레온: 검 공격'


In [20]:
# 연습 5 · 딥러닝 연결: 문장 · 라벨 Dataset
class TextDataset:                      # 문장 · 라벨 데이터셋
    def __init__(self, texts, labels):  # 문장 목록 · 라벨 목록
        self.texts = texts              # 문장
        self.labels = labels            # 라벨 (1 긍정 · 0 부정)

    def __len__(self):                                       # len(tds)
        raise NotImplementedError("TODO 1: 문장 개수 반환")  # 이 줄을 지우고 작성

    def __getitem__(self, i):                                               # tds[i]
        raise NotImplementedError("TODO 2: (i번째 문장, i번째 라벨) 반환")  # 이 줄을 지우고 작성


# 테스트 (수정하지 않기)
try:
    tds = TextDataset(["좋아요", "별로예요", "최고"], [1, 0, 1])  # 문장 3개
    assert len(tds) == 3, "__len__ 확인"                          # __len__ 검사
    assert tds[1] == ("별로예요", 0), "__getitem__ 확인"          # __getitem__ 검사
    # for 문 검사
    assert list(tds) == [("좋아요", 1), ("별로예요", 0), ("최고", 1)], "for 문으로 전부 꺼내기"
except (AssertionError, NotImplementedError) as e:                # 실패 · 미작성
    print("미완료 · 확인할 내용:", e)                             # 무엇이 틀렸는지
else:
    print("연습 5 통과")                                          # 모든 검사 통과

미완료 · 확인할 내용: TODO 1: 문장 개수 반환


### 3. 변형 문제 · 조건을 바꿔 다시 풀기

1. 2-2의 `Bag`을 고쳐 `x.items.append("apple")`이 `y.items`에 영향을 주지 않게 만들기 (2-2)
2. 3-3을 `SquareDataset(4)`로 바꾸면 `len(ds)` · `ds[3]` · `list(ds)`는? 예측 → 실행 (3-3)
3. 5장 모델을 `PetClassifier(num_classes=5)`로 만들면 파라미터 수와 4장 배치의 출력 모양은? (5장)

In [21]:
# 변형 문제 직접 작성

### 4. 모범 답안 · 1 ~ 3을 다 푼 뒤에 보기

#### 질문 답

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">섹션</th>
<th style="text-align: left;">답</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1-1</td>
<td style="text-align: left;">한쪽만 지우거나 섞으면 에러 없이 짝이 어긋남 → 한 덩어리로 묶어야 함</td>
</tr>
<tr>
<td style="text-align: left;">1-2</td>
<td style="text-align: left;"><code>__init__</code> — 넘긴 값을 <code>self.속성</code>으로 객체에 저장</td>
</tr>
<tr>
<td style="text-align: left;">2-1</td>
<td style="text-align: left;">메서드를 호출한 객체. <code>car1.m()</code>은 <code>Car.m(car1)</code>로 바뀌어 파이썬이 자동으로 넣음</td>
</tr>
<tr>
<td style="text-align: left;">2-2</td>
<td style="text-align: left;">객체마다 따로 vs 모두 공유. 공유 값은 <code>클래스.변수</code>로 바꾸고, 리스트는 <code>self.</code>로</td>
</tr>
<tr>
<td style="text-align: left;">3-1</td>
<td style="text-align: left;"><code>a.__add__(b)</code> · <code>x.__len__()</code> 같은 특수 메서드</td>
</tr>
<tr>
<td style="text-align: left;">3-2</td>
<td style="text-align: left;">print · f-string은 <code>__str__</code>, 리스트 안 · repr()은 <code>__repr__</code> (하나만이면 <code>__repr__</code>)</td>
</tr>
<tr>
<td style="text-align: left;">3-3</td>
<td style="text-align: left;"><code>__len__</code> (전체 개수) · <code>__getitem__</code> (i번째 한 쌍, 범위 밖은 IndexError)</td>
</tr>
<tr>
<td style="text-align: left;">3-4</td>
<td style="text-align: left;"><code>__call__</code> → <code>forward</code>. 직접 부르면 <code>__call__</code>의 공통 처리(hook 등)가 빠짐</td>
</tr>
<tr>
<td style="text-align: left;">4-1</td>
<td style="text-align: left;">부모 메서드를 그대로 씀 vs 같은 이름으로 다시 정의해 교체 (찾는 순서: 자식 → 부모)</td>
</tr>
<tr>
<td style="text-align: left;">4-2</td>
<td style="text-align: left;">같은 이름의 메서드 호출이 객체 종류에 따라 다르게 동작하는 것</td>
</tr>
<tr>
<td style="text-align: left;">4-3</td>
<td style="text-align: left;">부모가 만드는 속성이 없어 나중에 AttributeError, nn.Module은 층 저장 순간 바로 에러</td>
</tr>
<tr>
<td style="text-align: left;">5</td>
<td style="text-align: left;">Dataset: <code>__len__</code> · <code>__getitem__</code> / nn.Module: <code>super().__init__()</code>이 든 <code>__init__</code> · <code>forward</code></td>
</tr>
</tbody>
</table>

#### 코드 문제 답

TODO 부분만 바꾼 클래스 (테스트는 그대로 두고 함께 실행)

**연습 1**

```python
class MenuItem:
    def __init__(self, name, price):
        self.name = name
        self.price = price

    def apply_discount(self, rate):
        if not 0 <= rate <= 1:
            raise ValueError("할인율은 0 이상 1 이하")
        self.price = round(self.price * (1 - rate))

    def __str__(self):
        return f"{self.name}: {self.price:,}원"
```

**연습 2**

```python
class Student:
    total_created = 0

    def __init__(self, name, score):
        self.name = name
        self.score = score
        Student.total_created += 1  # self.total_created += 1 은 X

    def update_score(self, score):
        self.score = score
```

**연습 3**

```python
class Book:
    def __init__(self, title, author, price):
        self.title = title
        self.author = author
        self.price = price

    def __str__(self):
        return f"{self.title} - {self.author} / {self.price:,}원"

    def __repr__(self):
        return f"Book(title={self.title!r}, author={self.author!r}, price={self.price})"
```

**연습 4**

```python
class Warrior(Character):
    def attack(self):
        return f"{self.name}: 검 공격"


class Mage(Character):
    def attack(self):
        return f"{self.name}: 마법 공격"


class DoubleAttackWarrior(Warrior):
    def attack(self):
        return super().attack() + " + 추가 공격"
```

**연습 5**

```python
class TextDataset:
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        return self.texts[i], self.labels[i]  # 범위 밖이면 리스트가 IndexError
```

#### 변형 문제 답

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">문제</th>
<th style="text-align: left;">답</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;"><code>__init__</code>에서 <code>self.items = []</code> → 객체마다 새 리스트, <code>y.items</code>는 <code>[]</code> 그대로</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;"><code>4</code> · <code>(3, 9)</code> · <code>[(0, 0), (1, 1), (2, 4), (3, 9)]</code></td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">192 × 5 + 5 = 965개 · 출력 모양 <code>(4, 5)</code></td>
</tr>
</tbody>
</table>

---
## 오답 노트

틀린 문제는 정답을 베끼지 말고 **원인**을 적음. 다음 복습 때 이 표의 "재발 방지 질문"부터 스스로에게 물어보기.

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">틀린 내용</th>
<th style="text-align: left;">틀린 이유</th>
<th style="text-align: left;">올바른 원칙</th>
<th style="text-align: left;">재발 방지 질문</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">(예) <code>a.total_created = 100</code> 뒤 <code>Car.total_created</code>를 100으로 답함</td>
<td style="text-align: left;">객체 이름으로 대입해도 클래스 변수가 바뀐다고 착각</td>
<td style="text-align: left;">객체 대입 = 그 객체에 인스턴스 변수 생성</td>
<td style="text-align: left;">바꾼 곳이 <code>클래스.</code>인가 <code>객체.</code>인가?</td>
</tr>
<tr>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
</tr>
<tr>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
</tr>
</tbody>
</table>

---
## 면접 30초 답변

**Q1. 딕셔너리 대신 클래스를 쓰는 이유는 무엇인가요?**
> - 딕셔너리는 데이터만 묶고 키 오타는 실행해야 드러나며, 그 데이터를 다루는 함수는 따로 관리해야 합니다.
> - 클래스는 속성과 메서드를 한 설계도에 묶어, 같은 구조의 객체를 여러 개 만들고 값 검증 같은 규칙을 메서드 한 곳에서 지키게 합니다.
> - PyTorch의 `Dataset`과 모델이 클래스인 이유도 데이터와 가중치, 동작을 함께 다루기 위해서입니다.

**Q2. `self`는 무엇인가요?**
> - 메서드를 호출한 객체 자신입니다.
> - `car1.show()`를 부르면 파이썬이 `Car.show(car1)`로 바꿔 `car1`을 첫 인자 `self`에 넣습니다.
> - 그래서 `self.price`처럼 쓰면 그 객체에 저장된 값을 읽고 바꿀 수 있고, `self.`를 빠뜨리면 지역 변수가 되어 값이 남지 않습니다.

**Q3. PyTorch `Dataset`에서 `__len__`과 `__getitem__`은 왜 필요한가요?**
> - `DataLoader`가 이 두 메서드에 의존하기 때문입니다.
> - `len(dataset)`으로 전체 개수를 알아 섞을 번호와 배치 수를 정하고, `dataset[i]`로 샘플을 하나씩 받아 배치로 묶습니다.
> - `__getitem__`에서 그때그때 이미지를 읽으면 전체 데이터를 메모리에 올리지 않아도 됩니다.

**Q4. `nn.Module`을 상속할 때 `super().__init__()`을 먼저 호출하는 이유는?**
> 부모인 `nn.Module`이 층과 파라미터를 등록할 저장 공간을 `__init__`에서 준비하기 때문입니다. 이를 먼저 호출하지 않고 `self.fc = nn.Linear(...)`를 쓰면 `cannot assign module before Module.__init__() call` 에러가 나고, 호출해야 `parameters()`와 `to(device)`가 층을 찾을 수 있습니다.

**Q5. `model.forward(x)` 대신 `model(x)`로 호출하는 이유는?**
> `model(x)`는 `nn.Module`의 `__call__`을 거쳐 `forward`를 부르는데, `__call__`이 hook 같은 공통 처리를 함께 실행합니다. `forward`를 직접 부르면 계산 결과는 같아도 이 처리가 빠져, hook에 의존하는 도구가 제대로 동작하지 않을 수 있습니다.

---
## 최종 요약표

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">함수·문법</th>
<th style="text-align: left;">소속</th>
<th style="text-align: left;">역할</th>
<th style="text-align: left;">코드에서 사용한 위치</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>del</code> · <code>zip()</code> · <code>KeyError</code></td>
<td style="text-align: left;">Python 기본 문법 · 내장 함수</td>
<td style="text-align: left;">삭제 · 짝짓기 · 없는 키 에러</td>
<td style="text-align: left;">1-1 <code>del companies[1]</code> · <code>zip(companies, details)</code> · <code>except KeyError as e:</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>class</code> · <code>__init__</code></td>
<td style="text-align: left;">Python 기본 문법</td>
<td style="text-align: left;">설계도 정의 · 생성 때 자동 실행</td>
<td style="text-align: left;">1-2 <code>class Car:</code> · <code>def __init__(self, company, color, horsepower, price):</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>self.속성</code> · <code>객체.메서드()</code></td>
<td style="text-align: left;">Python 기본 문법</td>
<td style="text-align: left;">값 저장 · 기능 호출</td>
<td style="text-align: left;">1-2 <code>self.company = company</code> · <code>car.is_high_performance()</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>__dict__</code></td>
<td style="text-align: left;">object (모든 객체)</td>
<td style="text-align: left;">객체에 저장된 속성 보기</td>
<td style="text-align: left;">1-2 <code>car2.__dict__</code> · 2-2 <code>a.__dict__</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>raise ValueError()</code></td>
<td style="text-align: left;">Python 기본 문법</td>
<td style="text-align: left;">직접 에러 발생</td>
<td style="text-align: left;">1-2 <code>raise ValueError("할인율은 0 이상 1 이하")</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>self</code></td>
<td style="text-align: left;">Python 기본 문법 (관례 이름)</td>
<td style="text-align: left;">메서드를 호출한 객체</td>
<td style="text-align: left;">2-1 <code>def who(self): return self</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>클래스.메서드(객체)</code></td>
<td style="text-align: left;">Python 기본 문법</td>
<td style="text-align: left;">self를 직접 넣어 호출</td>
<td style="text-align: left;">2-1 <code>Car.is_high_performance(car1)</code></td>
</tr>
<tr>
<td style="text-align: left;">인스턴스 변수 · 클래스 변수</td>
<td style="text-align: left;">객체 · class</td>
<td style="text-align: left;">객체별 값 · 공유 값</td>
<td style="text-align: left;">2-2 <code>self.price = price</code> · <code>total_created = 0</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>del 객체.속성</code></td>
<td style="text-align: left;">Python 기본 문법</td>
<td style="text-align: left;">인스턴스 변수 삭제</td>
<td style="text-align: left;">2-2 <code>del a.total_created</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>type()</code> · <code>isinstance()</code></td>
<td style="text-align: left;">Python 내장 함수</td>
<td style="text-align: left;">클래스 확인 · 소속 확인</td>
<td style="text-align: left;">3-1 <code>type(10).__name__</code> · <code>isinstance(10, int)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>__add__</code> · <code>__mul__</code> · <code>__bool__</code> · <code>__contains__</code></td>
<td style="text-align: left;">특수 메서드</td>
<td style="text-align: left;"><code>+</code> · <code>*</code> · <code>bool()</code> · <code>in</code></td>
<td style="text-align: left;">3-1 <code>number.__add__(5)</code> · <code>number.__mul__(3)</code> · <code>number.__bool__()</code> · <code>x.__contains__(20)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>__str__</code> · <code>__repr__</code> · <code>repr()</code> · <code>!r</code></td>
<td style="text-align: left;">특수 메서드 · 내장 함수 · f-string</td>
<td style="text-align: left;">사용자용 · 개발자용 출력</td>
<td style="text-align: left;">3-2 <code>def __str__(self):</code> · <code>def __repr__(self):</code> · <code>repr(p1)</code> · <code>{self.name!r}</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>__len__</code> · <code>__getitem__</code></td>
<td style="text-align: left;">특수 메서드</td>
<td style="text-align: left;"><code>len()</code> · <code>[i]</code> · for 문</td>
<td style="text-align: left;">3-3 <code>def __len__(self):</code> · <code>def __getitem__(self, i):</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>__call__</code> · <code>callable()</code></td>
<td style="text-align: left;">특수 메서드 · 내장 함수</td>
<td style="text-align: left;"><code>객체(x)</code> 호출 · 호출 가능 여부</td>
<td style="text-align: left;">3-4 <code>def __call__(self, x):</code> · <code>callable(double)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>class 자식(부모):</code></td>
<td style="text-align: left;">Python 기본 문법</td>
<td style="text-align: left;">상속</td>
<td style="text-align: left;">4-1 <code>class Dog(Animal):</code></td>
</tr>
<tr>
<td style="text-align: left;">오버라이딩</td>
<td style="text-align: left;">Python 기본 문법</td>
<td style="text-align: left;">같은 이름으로 다시 정의</td>
<td style="text-align: left;">4-1 <code>def sound(self): return "멍멍"</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>object</code></td>
<td style="text-align: left;">Python 내장 클래스</td>
<td style="text-align: left;">모든 클래스의 맨 위 부모</td>
<td style="text-align: left;">4-1 <code>Dog.__mro__</code> → <code>['Dog', 'Animal', 'object']</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>__mro__</code> · <code>issubclass()</code></td>
<td style="text-align: left;">class · 내장 함수</td>
<td style="text-align: left;">메서드 찾는 순서 · 클래스 관계</td>
<td style="text-align: left;">4-1 <code>Dog.__mro__</code> · <code>issubclass(Dog, Animal)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>super()</code></td>
<td style="text-align: left;">Python 내장 함수</td>
<td style="text-align: left;">부모 메서드 호출</td>
<td style="text-align: left;">4-3 <code>super().sound() + "!!!"</code> · <code>super().__init__(name)</code> · 5 <code>super().__init__()</code></td>
</tr>
</tbody>
</table>

### 할 것 · 하지 말 것

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">✅ 할 것</th>
<th style="text-align: left;">⚠️ 하지 말 것</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">자식 <code>__init__</code> 첫 줄에 <code>super().__init__()</code></td>
<td style="text-align: left;"><code>model.forward(x)</code> 직접 호출</td>
</tr>
<tr>
<td style="text-align: left;">객체마다 다른 값은 <code>self.</code>에 저장</td>
<td style="text-align: left;">클래스 변수에 리스트 두기</td>
</tr>
<tr>
<td style="text-align: left;">모델은 <code>model(x)</code>로 호출</td>
<td style="text-align: left;"><code>self.</code> 빠뜨린 저장</td>
</tr>
<tr>
<td style="text-align: left;"><code>__getitem__</code>은 샘플 1개 기준 모양으로</td>
<td style="text-align: left;"><code>__getitem__</code>에서 배치 차원 직접 추가</td>
</tr>
<tr>
<td style="text-align: left;">출력용 메서드는 <code>__repr__</code> 먼저</td>
<td style="text-align: left;">속성을 밖에서 바꿔 검증 건너뛰기</td>
</tr>
</tbody>
</table>

### 헷갈리는 짝 한눈에 보기

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">짝</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">결론</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">클래스 vs 객체</td>
<td style="text-align: left;">설계도 vs 실물</td>
<td style="text-align: left;"><code>Car</code> 1개로 <code>car1</code> · <code>car2</code> 여러 개</td>
</tr>
<tr>
<td style="text-align: left;">속성 vs 메서드</td>
<td style="text-align: left;">값 (괄호 X) vs 기능 (괄호 O)</td>
<td style="text-align: left;"><code>car.price</code> · <code>car.apply_discount(0.1)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>self.x = x</code> vs <code>x = x</code></td>
<td style="text-align: left;">객체에 저장 vs 지역 변수</td>
<td style="text-align: left;">기억할 값은 <code>self.</code></td>
</tr>
<tr>
<td style="text-align: left;">인스턴스 변수 vs 클래스 변수</td>
<td style="text-align: left;">객체마다 vs 모두 공유</td>
<td style="text-align: left;">기본은 인스턴스 변수</td>
</tr>
<tr>
<td style="text-align: left;"><code>객체.변수 = 값</code> vs <code>클래스.변수 = 값</code></td>
<td style="text-align: left;">그 객체에 새로 생성 vs 공유 값 변경</td>
<td style="text-align: left;">공유 값은 <code>클래스.</code>로</td>
</tr>
<tr>
<td style="text-align: left;"><code>__str__</code> vs <code>__repr__</code></td>
<td style="text-align: left;">사용자용 vs 개발자용 (리스트 안)</td>
<td style="text-align: left;">하나만이면 <code>__repr__</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>__getitem__</code> vs <code>__iter__</code></td>
<td style="text-align: left;">번호로 꺼내기 vs 순서대로만</td>
<td style="text-align: left;">Dataset은 <code>__getitem__</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>model(x)</code> vs <code>model.forward(x)</code></td>
<td style="text-align: left;"><code>__call__</code>을 거침 vs 계산만</td>
<td style="text-align: left;"><code>model(x)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>type(x) is</code> vs <code>isinstance</code></td>
<td style="text-align: left;">정확히 그 클래스 vs 상속까지</td>
<td style="text-align: left;"><code>isinstance</code></td>
</tr>
<tr>
<td style="text-align: left;">오버라이딩 vs <code>super()</code></td>
<td style="text-align: left;">완전 교체 vs 부모 기능 + 추가</td>
<td style="text-align: left;">확장은 <code>super()</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>super().__init__()</code> 생략</td>
<td style="text-align: left;">일반 클래스는 나중에 에러 vs nn.Module은 즉시 에러</td>
<td style="text-align: left;">자식 <code>__init__</code> 첫 줄에 항상</td>
</tr>
</tbody>
</table>

---
## 다음 복습일

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">언제</th>
<th style="text-align: left;">할 일</th>
<th style="text-align: left;">걸리는 시간</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">내일</td>
<td style="text-align: left;">기억 확인 문제 1을 자료 없이 다시 답하고 오답 노트 갱신</td>
<td style="text-align: left;">15분</td>
</tr>
<tr>
<td style="text-align: left;">3일 뒤</td>
<td style="text-align: left;">변형 문제 3개를 새로 풀기 · 연습 1 ~ 5를 빈 칸부터 다시</td>
<td style="text-align: left;">30분</td>
</tr>
<tr>
<td style="text-align: left;">7일 뒤</td>
<td style="text-align: left;">5장 <code>PetDataset</code> · <code>PetClassifier</code>를 빈 셀에 직접 다시 짜고 출력 모양 비교</td>
<td style="text-align: left;">30분</td>
</tr>
<tr>
<td style="text-align: left;">14일 뒤</td>
<td style="text-align: left;">오답 노트에 남은 문제만 다시 풀기 · 03_NumPy 시작 전 ✅ 줄만 읽기</td>
<td style="text-align: left;">15분</td>
</tr>
</tbody>
</table>

---
## 부록

### 이 노트북을 만든 방법

- 수업 노트북 4개(`8월 6일(ipynb)/1 · 2 · 4 · 8`)를 주제별로 묶어 12개 섹션으로 만듦 (대응표 아래)
- 각 섹션은 정답 코드를 실행한 출력과 해설을 함께 저장 · 형식 v2.1

### 읽는 법

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">표시</th>
<th style="text-align: left;">의미</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>A</code> · <code>B</code></td>
<td style="text-align: left;"><strong>A</strong> 딥러닝 코드에 바로 나오는 필수 · <strong>B</strong> 알아두면 되는 것</td>
</tr>
<tr>
<td style="text-align: left;">🎯</td>
<td style="text-align: left;">이번 섹션에서 답해야 할 질문</td>
</tr>
<tr>
<td style="text-align: left;">💡</td>
<td style="text-align: left;">쉬운 설명 · 비유</td>
</tr>
<tr>
<td style="text-align: left;">⚠️</td>
<td style="text-align: left;">자주 하는 실수</td>
</tr>
<tr>
<td style="text-align: left;">✅</td>
<td style="text-align: left;">기억할 결론 = 🎯 질문의 답</td>
</tr>
</tbody>
</table>

**섹션 구성** · 🎯 질문 → 💡 쉬운 설명 → 실행 흐름 → 먼저 예측 → 코드 → 결과(예측 정답 · 출력 줄) → 차이 · 언제 쓰나 → ⚠️ 주의 → ✅ 한 줄 정리

**코드 주석** · 코드 위 = 이 단계에서 하는 일, 코드 옆 = 그 줄이 왜 필요한가 (줄마다 · 열 맞춤)

### 원본 노트북 → 섹션 대응표

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">원본 (수업 8월 6일)</th>
<th style="text-align: left;">섹션</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">01 클래스가 필요한 이유 (<code>1.ipynb</code>)</td>
<td style="text-align: left;">1-1 · 1-2 · 1-3 (<code>__str__</code> · <code>__repr__</code>은 3-2로 <strong>통합</strong>)</td>
</tr>
<tr>
<td style="text-align: left;">02 self · 인스턴스 변수 (<code>2.ipynb</code>)</td>
<td style="text-align: left;">2-1 · 2-2 (<code>isinstance</code>는 3-1 · 4-1로 <strong>통합</strong>)</td>
</tr>
<tr>
<td style="text-align: left;">03 특수 메서드 (<code>4.ipynb</code>)</td>
<td style="text-align: left;">3-1 · 3-2</td>
</tr>
<tr>
<td style="text-align: left;">04 상속 · super (<code>8.ipynb</code>)</td>
<td style="text-align: left;">4-1 · 4-2 · 4-3</td>
</tr>
<tr>
<td style="text-align: left;">보충 · 딥러닝 연결</td>
<td style="text-align: left;">3-3 <code>__len__</code> · <code>__getitem__</code>, 3-4 <code>__call__</code> (원본 03 특수 메서드의 보충), 5장 PyTorch</td>
</tr>
<tr>
<td style="text-align: left;">활동 <code>1-1</code> · <code>2-1</code> · <code>4-1</code> · <code>8-1</code></td>
<td style="text-align: left;">기억 확인 문제 2 · 연습 1 ~ 4 (연습 문제 재료로만 사용)</td>
</tr>
<tr>
<td style="text-align: left;">객체 참조 (<code>9.ipynb</code>)</td>
<td style="text-align: left;">제외 — 01_Python문법 1-2와 중복</td>
</tr>
</tbody>
</table>

원본의 수업 중 시험 셀(빈 셀 · `a.__sub__(b)` · `dir()` 출력)은 제외, `class Hoho(object)`의 뜻은 4-1 ⚠️에서 설명

### 전체 지도

#### 단원 연결 · 왜 이 순서로 배우는가

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: center;">순서</th>
<th style="text-align: left;">배우는 이유</th>
<th style="text-align: left;">대단원</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: center;">1</td>
<td style="text-align: left;">흩어진 데이터와 기능을 한 덩어리로 묶는다</td>
<td style="text-align: left;">클래스 · 객체</td>
</tr>
<tr>
<td style="text-align: center;">2</td>
<td style="text-align: left;">메서드가 어느 객체의 값을 쓰는지 정한다</td>
<td style="text-align: left;">self · 변수</td>
</tr>
<tr>
<td style="text-align: center;">3</td>
<td style="text-align: left;"><code>len()</code> · <code>[i]</code> · <code>print()</code> · <code>객체(x)</code>를 내 객체에 연결한다</td>
<td style="text-align: left;">특수 메서드</td>
</tr>
<tr>
<td style="text-align: center;">4</td>
<td style="text-align: left;">이미 있는 클래스를 물려받아 필요한 부분만 고친다</td>
<td style="text-align: left;">상속 · super</td>
</tr>
<tr>
<td style="text-align: center;">5</td>
<td style="text-align: left;">1~4를 합쳐 PyTorch Dataset · 모델을 만든다</td>
<td style="text-align: left;">종합 실습</td>
</tr>
</tbody>
</table>

#### ML/DL 공통 파이프라인 · 각 대단원이 쓰이는 곳

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">단계</th>
<th style="text-align: left;">하는 일</th>
<th style="text-align: left;">쓰이는 장</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">① 데이터 준비</td>
<td style="text-align: left;">파일 · 라벨 목록 만들기</td>
<td style="text-align: left;">01_Python문법</td>
</tr>
<tr>
<td style="text-align: left;">② 불러오기</td>
<td style="text-align: left;"><code>Dataset</code> · <code>DataLoader</code></td>
<td style="text-align: left;">2장 · 3장 · 4장 · <strong>5장</strong></td>
</tr>
<tr>
<td style="text-align: left;">③ 전처리</td>
<td style="text-align: left;"><code>__getitem__</code> 안에서 이미지 변환</td>
<td style="text-align: left;">3장 · <strong>5장</strong></td>
</tr>
<tr>
<td style="text-align: left;">④ 모델</td>
<td style="text-align: left;"><code>nn.Module</code> 상속, 층을 <code>self.</code>에 저장</td>
<td style="text-align: left;">1장 · 2장 · 4장 · <strong>5장</strong></td>
</tr>
<tr>
<td style="text-align: left;">⑤ 학습</td>
<td style="text-align: left;"><code>model(x)</code>, <code>optimizer.step()</code></td>
<td style="text-align: left;">1장 · 3장 · <strong>5장</strong></td>
</tr>
<tr>
<td style="text-align: left;">⑥ 평가</td>
<td style="text-align: left;"><code>model.eval()</code> 등 물려받은 메서드</td>
<td style="text-align: left;">4장</td>
</tr>
<tr>
<td style="text-align: left;">⑦ 저장 · 기록</td>
<td style="text-align: left;"><code>print(model)</code>, <code>model.state_dict()</code></td>
<td style="text-align: left;">3장 · 4장</td>
</tr>
</tbody>
</table>

**굵은 5장** · 종합 실습이 ② ~ ⑤ 를 한 번에 연결

### 전체 코드

1 ~ 5장 정답 코드를 장별 셀로 모음. **위에서부터 이 셀들만 실행해도 전체가 동작**하며, 각 출력 앞 `[번호]`가 섹션.

In [22]:
# ============================================================
# Python 클래스 전체 정리 — 전체 코드
# ============================================================
import torch                                      # 텐서
from torch import nn                              # 층 · 모델
from torch.utils.data import Dataset, DataLoader  # 데이터셋 부모 · 배치 공급

In [23]:
# ---------- 1. 클래스가 필요한 이유 ----------
# 1-1 리스트 두 개로 나눠 저장하면 한쪽만 지웠을 때 짝이 어긋남
companies, colors = ["Ferrari", "BMW", "Audi"], ["White", "Black", "Silver"]
del companies[1]                              # 한쪽 리스트에서만 삭제
print("[1-1]", list(zip(companies, colors)))  # Audi ↔ Black (BMW 색)


# 1-2 클래스 = 데이터(속성) + 기능(메서드)
class Car:                                           # 설계도 이름 (대문자로 시작)
    def __init__(self, company, horsepower, price):  # Car(...) 할 때 자동 실행
        self.company = company                       # 객체에 저장
        self.horsepower = horsepower                 # 마력
        self.price = price                           # 가격

    def is_high_performance(self):     # 상태로 계산하는 메서드
        return self.horsepower >= 300  # 300마력 이상이면 True

    def apply_discount(self, rate):                     # 할인율 받기
        if not 0 <= rate <= 1:                          # 할인율 검증
            raise ValueError("할인율은 0 이상 1 이하")  # 잘못된 값은 여기서 멈춤
        self.price = round(self.price * (1 - rate))     # 이 객체의 가격만 변경


car1, car2 = Car("Ferrari", 400, 8000), Car("BMW", 270, 5000)  # 같은 설계도로 객체 둘
print("[1-2]", car1.__dict__, car2.is_high_performance())      # 저장된 값 · 메서드 결과

# 1-3 메서드로 상태 변경: 그 객체만 바뀜
car1.apply_discount(0.1)                # car1만 10% 할인
print("[1-3]", car1.price, car2.price)  # car2는 그대로

[1-1] [('Ferrari', 'White'), ('Audi', 'Black')]
[1-2] {'company': 'Ferrari', 'horsepower': 400, 'price': 8000} False
[1-3] 7200 5000


In [24]:
# ---------- 2. self · 인스턴스 변수 · 클래스 변수 ----------
# 2-1 객체.메서드() = 클래스.메서드(객체)
print("[2-1]", car1.is_high_performance() == Car.is_high_performance(car1))


# 2-2 클래스 변수는 공유, 객체 이름으로 대입하면 인스턴스 변수가 새로 생김
class Counted:         # 생성 수를 세는 클래스
    total_created = 0  # 클래스 변수 (공유)

    def __init__(self):             # super().__init__() 없음
        Counted.total_created += 1  # 클래스 이름으로 변경


a, b = Counted(), Counted()                                              # 생성 2번
a.total_created = 100                                                    # 객체 이름으로 대입
print("[2-2]", Counted.total_created, a.total_created, b.total_created)  # a만 100

[2-1] True
[2-2] 2 100 2


In [25]:
# ---------- 3. 특수 메서드 ----------
# 3-1 연산자 · len() · [ ] = 특수 메서드 호출
print("[3-1]", 10 + 5 == (10).__add__(5), len([1, 2]) == [1, 2].__len__())


# 3-2 __str__ (사용자용) · __repr__ (개발자용)
class Product:                        # 출력 방법을 정한 클래스
    def __init__(self, name, price):  # 이름 · 가격 받기
        self.name = name              # 상품 이름
        self.price = price            # 가격

    def __str__(self):                           # print(메뉴)에 쓰임
        return f"{self.name}: {self.price:,}원"  # 천 단위 쉼표로 읽기 쉽게

    def __repr__(self):                                            # __str__ 없음
        return f"Product(name={self.name!r}, price={self.price})"  # 개발자용 표현


p1 = Product("Keyboard", 35000)  # 상품 1
print("[3-2]", p1, "|", [p1])    # __str__ · 리스트 안 __repr__


# 3-3 __len__ · __getitem__ → len(ds) · ds[i] · for 문
class SquareDataset:        # Dataset 원리를 보는 작은 클래스
    def __init__(self, n):  # n = 데이터 개수
        self.n = n          # 개수만 저장

    def __len__(self):  # len(tds)
        return self.n   # 저장해 둔 개수

    def __getitem__(self, i):                 # tds[i]
        if not 0 <= i < self.n:               # 0 ~ n-1 밖이면
            raise IndexError("범위 밖 번호")  # for 문이 멈추는 신호
        return i, i * i                       # 요청받은 순간 계산


sq = SquareDataset(5)                     # 5개짜리 데이터셋
print("[3-3]", len(sq), sq[3], list(sq))  # len · [i] · for 문


# 3-4 __call__ → 객체(x)가 forward를 부름
class Scale:                # 층(layer)을 흉내 낸 클래스
    def __init__(self, k):  # k = 곱할 값
        self.k = k          # 설정값 기억

    def __call__(self, x):      # 객체(x)가 부르는 메서드
        return self.forward(x)  # 계산은 forward에

    def forward(self, x):  # 실제 계산
        return x * self.k  # 입력 × 설정값


print("[3-4]", Scale(2)(5), callable(Scale(2)))  # 객체를 함수처럼

[3-1] True True
[3-2] Keyboard: 35,000원 | [Product(name='Keyboard', price=35000)]
[3-3] 5 (3, 9) [(0, 0), (1, 1), (2, 4), (3, 9), (4, 16)]
[3-4] 10 True


In [26]:
# ---------- 4. 상속 · 다형성 · super ----------
# 4-1 상속 · 오버라이딩
class Animal:                  # 부모 클래스
    def __init__(self, name):  # 이름을 받지만
        self.name = name       # 상품 이름

    def introduce(self):                        # 자식이 그대로 쓸 메서드
        return f"제 이름은 {self.name}입니다."  # 이름으로 소개 문장

    def sound(self):                    # 자식이 바꿀 메서드
        return "동물이 소리를 냅니다."  # 기본 소리


class Dog(Animal):     # Animal을 물려받음
    def sound(self):   # 자식이 바꿀 메서드
        return "멍멍"  # Dog만의 소리


class Cat(Animal):     # Animal을 물려받음
    def sound(self):   # 자식이 바꿀 메서드
        return "야옹"  # Cat만의 소리


print("[4-1]", Dog("콩이").introduce(), isinstance(Dog("콩이"), Animal))  # 상속 확인

# 4-2 다형성: 같은 호출 → 객체마다 다른 결과
# 같은 호출 · 다른 결과
print("[4-2]", [x.sound() for x in [Animal("동물"), Dog("초코"), Cat("나비")]])


# 4-3 super(): 부모 기능 + 추가
class Bird(Animal):                     # Animal을 물려받음
    def __init__(self, name, can_fly):  # 자식은 인자가 하나 더
        super().__init__(name)          # 부모가 name 저장
        self.can_fly = can_fly          # 자식만의 속성


print("[4-3]", Bird("참새", True).__dict__)  # name · can_fly 둘 다

[4-1] 제 이름은 콩이입니다. True
[4-2] ['동물이 소리를 냅니다.', '멍멍', '야옹']
[4-3] {'name': '참새', 'can_fly': True}


In [27]:
# ---------- 5. 종합 실습: Dataset · nn.Module ----------
# 5-1 Dataset 상속 + __len__ · __getitem__ → DataLoader로 배치
counts = {"bird": 3, "cat": 5, "dog": 4}                           # 클래스별 가짜 이미지 수
class_to_idx = {name: i for i, name in enumerate(sorted(counts))}  # 이름순 번호
items = [(f"pets/{name}/img_{i:03d}.jpg", class_to_idx[name])      # (경로, 번호) 쌍
         for name, n in counts.items() for i in range(n)]


class PetDataset(Dataset):              # Dataset을 물려받음
    def __init__(self, items, size=8):  # 목록 · 이미지 크기
        self.items = items              # (경로, 번호) 목록
        self.size = size                # 가짜 이미지 한 변

    def __len__(self):          # len(tds)
        return len(self.items)  # 목록 길이 = 데이터 수

    def __getitem__(self, i):                                                # tds[i]
        path, target = self.items[i]                                         # i번째 경로 · 번호
        return torch.full((3, self.size, self.size), float(target)), target  # (가짜 이미지, 번호)


loader = DataLoader(PetDataset(items), batch_size=4, shuffle=False)  # 4개씩 묶기
print("[5-1]", [targets.tolist() for _, targets in loader])          # 배치별 번호


# 5-2 nn.Module 상속 + super().__init__() + forward → model(x)
class PetClassifier(nn.Module):                            # nn.Module을 물려받은 모델
    def __init__(self, num_classes=3, size=8):             # 클래스 수 · 이미지 크기
        super().__init__()                                 # nn.Module 준비를 먼저
        self.flatten = nn.Flatten()                        # (N, 3, 8, 8) → (N, 192)
        self.fc = nn.Linear(3 * size * size, num_classes)  # 192 → 3

    def forward(self, x):                # 실제 계산
        return self.fc(self.flatten(x))  # 펴기 → 점수 3개


torch.manual_seed(0)            # 가중치 초기값 고정
model = PetClassifier()         # 모델 객체
images, _ = next(iter(loader))  # 첫 배치만
# 출력 모양 · 파라미터 수
print("[5-2]", tuple(model(images).shape), sum(p.numel() for p in model.parameters()))

[5-1] [[0, 0, 0, 1], [1, 1, 1, 1], [2, 2, 2, 2]]
[5-2] (4, 3) 579
